In [ ]:


import os
import re
import gc
import sys
import json
import math
import time
import random
import shutil
import warnings

import numpy as np
import pandas as pd

from PIL import Image

import torch

from torch.utils.data import (
    Dataset,
    DataLoader
)

from tqdm import tqdm

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    f1_score,
    balanced_accuracy_score
)

from huggingface_hub import whoami

from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    get_linear_schedule_with_warmup
)

import torch.nn as nn

import transformers


# ==================================================================================================
# 1. CLEAN GPU
# ==================================================================================================

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ==================================================================================================
# ==================================================================================================
# 2. PATHS
# ==================================================================================================

TRAINING_METADATA_CSV = (
    r"C:\Users\ga_sd360\OneDrive - The University of Akron"
    r"\Desktop\Fine Tuning\Patient 08\A1\Training"
    r"\Patient_08_A1_training_metadata.csv"
)

TESTING_METADATA_CSV = (
    r"C:\Users\ga_sd360\OneDrive - The University of Akron\Desktop\Fine Tuning\Patient 08\A1\Testing\Blinded\Patient_08_A1_blinded_test_labels.csv"
)

# IMPORTANT FOR WINDOWS / ONEDRIVE:
# Keep the output path deliberately SHORT. Windows can raise WinError 206
# when deeply nested paths approach the legacy MAX_PATH limit.
OUTPUT_BASE = (
    r"C:\Users\ga_sd360\OneDrive - The University of Akron"
    r"\Desktop\Fine Tuning\Patient 08\A1\LR08A1R4"
)

# Every execution gets a short fresh run folder.
RUN_ID = time.strftime("%y%m%d_%H%M%S")
OUTPUT_ROOT = os.path.join(OUTPUT_BASE, RUN_ID)

# Short checkpoint folder names keep every generated path safely below
# Windows path-length limits. No old directory is deleted or overwritten.
BEST_CHECKPOINT_ROOT = os.path.join(OUTPUT_ROOT, "ckpt")
os.makedirs(BEST_CHECKPOINT_ROOT, exist_ok=True)

# These are assigned after a best epoch is selected.
BEST_ADAPTER_DIR = None
BEST_REFT_WEIGHTS = None
BEST_REFT_CONFIG = None

TRAIN_SPLIT_CSV = os.path.join(OUTPUT_ROOT, "train.csv")
VALIDATION_SPLIT_CSV = os.path.join(OUTPUT_ROOT, "val.csv")
TRAINING_HISTORY_CSV = os.path.join(OUTPUT_ROOT, "history.csv")
FINAL_PREDICTIONS_CSV = os.path.join(OUTPUT_ROOT, "test_predictions.csv")
FINAL_METRICS_JSON = os.path.join(OUTPUT_ROOT, "metrics.json")
EXPERIMENT_CONFIG_JSON = os.path.join(OUTPUT_ROOT, "config.json")
TARGET_MODULES_TXT = os.path.join(OUTPUT_ROOT, "target.txt")
CONFUSION_MATRIX_CSV = os.path.join(OUTPUT_ROOT, "confusion.csv")

os.makedirs(OUTPUT_ROOT, exist_ok=True)

# 3. MODEL
# ==================================================================================================

MODEL_ID = "google/medgemma-4b-it"


# ==================================================================================================
# 4. EXACT SYSTEM PROMPT
# DO NOT CHANGE
# ==================================================================================================

SYSTEM_PROMPT = """
You are a strict vision-language classifier for EEG seizure detection.
Classify each EEG topomap image into exactly one of two labels:
Seizure
Non-seizure
Return only the final label. Do not provide explanation, puntuation, confidence score or extra text.
""".strip()


# ==================================================================================================
# 5. EXACT USER PROMPT
# DO NOT CHANGE
# ==================================================================================================

USER_PROMPT = """
You are an expert in identifying seizure patterns from EEG A1-coefficient variance topomaps. The attached image represents a 2-second EEG segment.
Classify the provided fixed-scale EEG A1-coefficient variance topomap as one of the following:
seizure
non-seizure
The raw EEG signal was divided into 2-second windows, and each window was filtered using a 60 Hz notch filter. A Discrete Wavelet Transform (DWT) was then applied separately to every bipolar EEG channel within each window, and the A1 approximation coefficients were extracted.
For each window, the variance of the A1 coefficients was calculated separately for every bipolar EEG channel. Each channel’s A1-coefficient variance value was positioned at the midpoint between the two electrodes forming that bipolar channel. These values were spatially interpolated to create the EEG topomap. All topomaps use the same fixed color scale, allowing direct comparison across images. Yellow regions indicate higher A1-coefficient variance, while dark purple regions indicate lower A1-coefficient variance.
Classify the image based on the visible A1-coefficient variance intensity and its spatial distribution. Consider whether high-variance activity is focal or widespread and whether the pattern is symmetric or asymmetric. Evaluate these characteristics together rather than using any single feature as a definitive rule.
""".strip()


# ==================================================================================================
# 6. RANDOM SEED
# ==================================================================================================

RANDOM_SEED = 42


def set_all_seeds(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed(seed)

        torch.cuda.manual_seed_all(seed)


set_all_seeds(
    RANDOM_SEED
)


# ==================================================================================================
# ==================================================================================================
# 7. LoReFT R4 CONFIGURATION
# ==================================================================================================

LOREFT_RANK = 4

# Four independent LoReFT interventions distributed across the 34-layer
# MedGemma language stack. Human layer numbers are 1-based.
LOREFT_TARGET_LAYERS_HUMAN = [8, 17, 26, 31]
NUM_LOREFT_INTERVENTIONS = len(LOREFT_TARGET_LAYERS_HUMAN)

LOREFT_PREFIX_TOKENS = 5
LOREFT_SUFFIX_TOKENS = 5
LOREFT_DROPOUT = 0.0
LOREFT_COMPONENT = "language transformer block output / residual representation"

# 8. TRAINING SETTINGS
# SAME AS BASELINE
# ==================================================================================================

NUM_EPOCHS = 50

# Maximum consecutive epochs allowed without a new best validation checkpoint.
EARLY_STOPPING_PATIENCE = 7

LEARNING_RATE = 9e-4

WEIGHT_DECAY = 0.01

TRAIN_BATCH_SIZE = 1

VALIDATION_BATCH_SIZE = 1

GRADIENT_ACCUMULATION_STEPS = 8

MAX_GRAD_NORM = 1.0

WARMUP_RATIO = 0.05

MAX_NEW_TOKENS = 8

NUM_WORKERS = 0


# ==================================================================================================
# 9. EXPECTED DATASET COUNTS
# ==================================================================================================
#
# Patient 08 A1 has:
#   - 1000 images in the training-pool CSV
#   - 1000 images in the separate blinded-test CSV
#
# The training pool is split deterministically and STRATIFIED by class:
#   - 80% actual training
#   - 20% validation
#
# Therefore:
#   - actual training total = 800
#   - validation total      = 200
#
# The exact seizure/non-seizure counts are derived from the CSV so the split
# preserves Patient 08's real class distribution rather than assuming a label ratio.
# ==================================================================================================

EXPECTED_FULL_TRAIN_TOTAL = 1000
EXPECTED_ACTUAL_TRAIN_TOTAL = 800
EXPECTED_VALIDATION_TOTAL = 200
EXPECTED_TEST_TOTAL = 1000

TRAIN_FRACTION = 0.80
VALIDATION_FRACTION = 0.20


# ==================================================================================================
# 10. HUGGING FACE LOGIN CHECK

# ==================================================================================================

print("\n" + "=" * 100)
print("HUGGING FACE LOGIN CHECK")
print("=" * 100)


try:

    hf_info = whoami()

    print(
        "\nLogged in as:",
        hf_info["name"]
    )


except Exception as error:

    raise RuntimeError(
        "\nHugging Face authentication is not available.\n"
        "Run huggingface_hub.login() first."
    ) from error


# ==================================================================================================
# 11. GPU CHECK
# ==================================================================================================

print("\n" + "=" * 100)
print("GPU CHECK")
print("=" * 100)


if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA GPU is not available."
    )


DEVICE = torch.device(
    "cuda:0"
)


GPU_NAME = torch.cuda.get_device_name(
    0
)


GPU_MEMORY_GB = (

    torch.cuda.get_device_properties(
        0
    ).total_memory

    /

    1024 ** 3
)


print(
    f"\nGPU: {GPU_NAME}"
)

print(
    f"GPU memory: {GPU_MEMORY_GB:.2f} GB"
)


# ==================================================================================================
# 12. BF16 / FP16
# ==================================================================================================

if torch.cuda.is_bf16_supported():

    MODEL_DTYPE = torch.bfloat16

    PRECISION_NAME = "bfloat16"


else:

    MODEL_DTYPE = torch.float16

    PRECISION_NAME = "float16"


print(
    f"Model/training dtype: {PRECISION_NAME}"
)


torch.backends.cuda.matmul.allow_tf32 = True


# ==================================================================================================
# 13. VERSION INFORMATION
# ==================================================================================================

print("\n" + "=" * 100)
print("PACKAGE VERSIONS")
print("=" * 100)


print(
    f"Python       : {sys.version.split()[0]}"
)

print(
    f"PyTorch      : {torch.__version__}"
)

print(
    f"Transformers : {transformers.__version__}"
)


# ==================================================================================================
# 14. PATH CLEANING
# ==================================================================================================

def clean_path(value):

    if pd.isna(value):

        return ""


    value = str(
        value
    ).strip()


    value = value.strip(
        '"'
    )

    value = value.strip(
        "'"
    )


    value = os.path.expandvars(
        value
    )

    value = os.path.expanduser(
        value
    )

    value = os.path.normpath(
        value
    )


    return value


# ==================================================================================================
# 15. IMAGE PATH COLUMN DETECTION
# ==================================================================================================

def detect_image_path_column(
    df,
    dataset_type
):

    if dataset_type == "test":

        candidates = [

            "blinded_image_path",
            "blinded_topomap_path",
            "blinded_path",

            "fine_tuning_image_path",

            "image_path",
            "selected_image_path",
            "original_image_path",

            "file_path",
            "filepath",
            "path"

        ]


    else:

        candidates = [

            "fine_tuning_image_path",

            "image_path",
            "selected_image_path",

            "blinded_image_path",
            "blinded_topomap_path",
            "blinded_path",

            "original_image_path",

            "file_path",
            "filepath",
            "path"

        ]


    for candidate in candidates:

        if candidate in df.columns:

            return candidate


    for column in df.columns:

        lower = str(
            column
        ).lower()


        if (
            "path" in lower

            and

            (
                "image" in lower
                or
                "topomap" in lower
                or
                "blinded" in lower
            )
        ):

            return column


    raise ValueError(
        "\nCould not automatically detect image path column.\n"
        f"Columns: {list(df.columns)}"
    )


# ==================================================================================================
# 16. LABEL COLUMN DETECTION
# ==================================================================================================

def detect_label_column(df):

    candidates = [

        "standard_label",

        "label",
        "Label",

        "target",

        "class_label",

        "class_name",

        "fine_tuning_class",

        "class",

        "category"

    ]


    for candidate in candidates:

        if candidate in df.columns:

            return candidate


    for column in df.columns:

        lower = str(
            column
        ).lower()


        if (
            "label" in lower
            or
            lower == "class"
        ):

            return column


    raise ValueError(
        "\nCould not automatically detect label column.\n"
        f"Columns: {list(df.columns)}"
    )


# ==================================================================================================
# 17. NORMALIZE LABELS
#
# Seizure     = 1
# Non-seizure = 0
# ==================================================================================================

def normalize_ground_truth_label(value):

    if pd.isna(value):

        return None


    if isinstance(
        value,
        (
            int,
            np.integer
        )
    ):

        if int(value) == 1:
            return 1

        if int(value) == 0:
            return 0


    if isinstance(
        value,
        (
            float,
            np.floating
        )
    ):

        if float(value) == 1.0:
            return 1

        if float(value) == 0.0:
            return 0


    text = str(
        value
    ).strip().lower()


    text = text.replace(
        "_",
        " "
    )


    text = text.replace(
        "–",
        "-"
    )


    text = re.sub(
        r"\s+",
        " ",
        text
    )


    # IMPORTANT:
    # Non-seizure is checked FIRST.

    if text in {

        "0",
        "0.0",

        "non seizure",
        "non-seizure",
        "nonseizure",

        "normal",
        "interictal",
        "ns"

    }:

        return 0


    if (
        "non" in text
        and
        "seizure" in text
    ):

        return 0


    if text in {

        "1",
        "1.0",

        "seizure",
        "ictal",
        "sz"

    }:

        return 1


    if "seizure" in text:

        return 1


    return None


# ==================================================================================================
# 18. LOAD + STANDARDIZE METADATA
# ==================================================================================================

def load_and_standardize_metadata(
    csv_path,
    dataset_name,
    dataset_type
):

    print("\n" + "=" * 100)

    print(
        f"LOADING {dataset_name.upper()}"
    )

    print("=" * 100)


    if not os.path.isfile(
        csv_path
    ):

        raise FileNotFoundError(
            f"\nCSV does not exist:\n{csv_path}"
        )


    df = pd.read_csv(
        csv_path
    )


    print(
        f"\nRows: {len(df)}"
    )


    print("\nColumns:")


    for column in df.columns:

        print(
            "  ",
            column
        )


    image_column = detect_image_path_column(
        df,
        dataset_type
    )


    label_column = detect_label_column(
        df
    )


    print(
        f"\nImage path column: {image_column}"
    )


    print(
        f"Label column     : {label_column}"
    )


    df["_image_path"] = df[
        image_column
    ].apply(
        clean_path
    )


    df["_label"] = df[
        label_column
    ].apply(
        normalize_ground_truth_label
    )


    invalid_labels = df[
        df["_label"].isna()
    ]


    if len(
        invalid_labels
    ) > 0:

        print(
            invalid_labels[
                [
                    label_column
                ]
            ]
            .drop_duplicates()
        )


        raise ValueError(
            "Some labels could not be interpreted."
        )


    df["_label"] = df[
        "_label"
    ].astype(
        int
    )


    # ----------------------------------------------------------------------------------------------
    # IMAGE EXISTENCE CHECK
    # ----------------------------------------------------------------------------------------------

    missing_images = []


    for image_path in df[
        "_image_path"
    ]:

        if not os.path.isfile(
            image_path
        ):

            missing_images.append(
                image_path
            )


    if missing_images:

        print(
            f"\nMissing images: {len(missing_images)}"
        )


        for image_path in missing_images[
            :20
        ]:

            print(
                image_path
            )


        raise FileNotFoundError(
            "One or more images are missing."
        )


    # ----------------------------------------------------------------------------------------------
    # DUPLICATE CURRENT PATH CHECK
    # ----------------------------------------------------------------------------------------------

    duplicate_count = int(

        df[
            "_image_path"
        ].duplicated().sum()
    )


    if duplicate_count > 0:

        raise ValueError(
            f"{duplicate_count} duplicate image paths detected."
        )


    seizure_count = int(

        (
            df["_label"] == 1
        ).sum()
    )


    non_seizure_count = int(

        (
            df["_label"] == 0
        ).sum()
    )


    print("\nClass counts:")


    print(
        f"Seizure     : {seizure_count}"
    )

    print(
        f"Non-seizure : {non_seizure_count}"
    )

    print(
        f"Total        : {len(df)}"
    )


    print(
        f"\nAll {len(df)} image paths exist."
    )


    return (
        df,
        image_column,
        label_column
    )


# ==================================================================================================
# 19. LOAD FULL TRAINING POOL
# ==================================================================================================

(
    full_training_df,
    train_image_column,
    train_label_column

) = load_and_standardize_metadata(

    TRAINING_METADATA_CSV,

    "Patient 08 A1 training pool",

    "train"
)


# ==================================================================================================
# 20. VERIFY FULL TRAINING COUNTS
# ==================================================================================================

full_training_seizure_count = int(
    (
        full_training_df["_label"] == 1
    ).sum()
)

full_training_non_seizure_count = int(
    (
        full_training_df["_label"] == 0
    ).sum()
)

if len(full_training_df) != EXPECTED_FULL_TRAIN_TOTAL:
    raise AssertionError(
        "Patient 08 training CSV must contain exactly "
        f"{EXPECTED_FULL_TRAIN_TOTAL} rows, but found {len(full_training_df)}."
    )

if (
    full_training_seizure_count
    + full_training_non_seizure_count
    != EXPECTED_FULL_TRAIN_TOTAL
):
    raise AssertionError(
        "Patient 08 training labels do not sum to the expected total."
    )

print(
    "\nVerified Patient 08 A1 full training pool: "
    f"{full_training_seizure_count} seizure + "
    f"{full_training_non_seizure_count} non-seizure = "
    f"{len(full_training_df)}"
)


# ==================================================================================================
# 21. CREATE THE EXACT SAME TRAIN / VALIDATION SPLIT

# RANDOM SEED = 42
# 80% TRAIN / 20% VALIDATION WITHIN EACH CLASS
# ==================================================================================================

seizure_pool = full_training_df[
    full_training_df["_label"] == 1
].copy()

non_seizure_pool = full_training_df[
    full_training_df["_label"] == 0
].copy()

seizure_pool = seizure_pool.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(
    drop=True
)

non_seizure_pool = non_seizure_pool.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(
    drop=True
)

# --------------------------------------------------------------------------------------------------
# STRATIFIED 80/20 COUNTS
# --------------------------------------------------------------------------------------------------
#
# Compute the training count independently inside each class.
# This preserves the class ratio of the 1000-image Patient 08 A1 training pool.
#
train_seizure_count = int(
    round(
        len(seizure_pool)
        * TRAIN_FRACTION
    )
)

train_non_seizure_count = int(
    round(
        len(non_seizure_pool)
        * TRAIN_FRACTION
    )
)

validation_seizure_count = (
    len(seizure_pool)
    - train_seizure_count
)

validation_non_seizure_count = (
    len(non_seizure_pool)
    - train_non_seizure_count
)

# Guard the requested exact 800 / 200 total split.
if (
    train_seizure_count
    + train_non_seizure_count
    != EXPECTED_ACTUAL_TRAIN_TOTAL
):
    raise AssertionError(
        "The class-stratified 80% split did not produce exactly "
        f"{EXPECTED_ACTUAL_TRAIN_TOTAL} training images. "
        "Check Patient 08 class counts."
    )

if (
    validation_seizure_count
    + validation_non_seizure_count
    != EXPECTED_VALIDATION_TOTAL
):
    raise AssertionError(
        "The class-stratified 20% split did not produce exactly "
        f"{EXPECTED_VALIDATION_TOTAL} validation images. "
        "Check Patient 08 class counts."
    )

# --------------------------------------------------------------------------------------------------
# TRAIN
# --------------------------------------------------------------------------------------------------

actual_train_seizure = seizure_pool.iloc[
    :train_seizure_count
].copy()

actual_train_non_seizure = non_seizure_pool.iloc[
    :train_non_seizure_count
].copy()

# --------------------------------------------------------------------------------------------------
# VALIDATION
# --------------------------------------------------------------------------------------------------

validation_seizure = seizure_pool.iloc[
    train_seizure_count:
].copy()

validation_non_seizure = non_seizure_pool.iloc[
    train_non_seizure_count:
].copy()

actual_train_df = pd.concat(
    [
        actual_train_seizure,
        actual_train_non_seizure
    ],
    ignore_index=True
)

validation_df = pd.concat(
    [
        validation_seizure,
        validation_non_seizure
    ],
    ignore_index=True
)

# Same deterministic final shuffle.
actual_train_df = actual_train_df.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(
    drop=True
)

validation_df = validation_df.sample(
    frac=1,
    random_state=RANDOM_SEED
).reset_index(
    drop=True
)


# ==================================================================================================
# 22. VERIFY SPLIT COUNTS

# ==================================================================================================

train_s = int(
    (
        actual_train_df["_label"] == 1
    ).sum()
)


train_ns = int(
    (
        actual_train_df["_label"] == 0
    ).sum()
)


val_s = int(
    (
        validation_df["_label"] == 1
    ).sum()
)


val_ns = int(
    (
        validation_df["_label"] == 0
    ).sum()
)


assert train_s == train_seizure_count

assert train_ns == train_non_seizure_count

assert val_s == validation_seizure_count

assert val_ns == validation_non_seizure_count

assert len(actual_train_df) == EXPECTED_ACTUAL_TRAIN_TOTAL

assert len(validation_df) == EXPECTED_VALIDATION_TOTAL


train_validation_overlap = set(

    actual_train_df[
        "_image_path"
    ]

).intersection(

    set(
        validation_df[
            "_image_path"
        ]
    )

)


assert len(
    train_validation_overlap
) == 0


print("\n" + "=" * 100)
print("PATIENT 08 A1 TRAIN / VALIDATION SPLIT")
print("=" * 100)


print("\nTRAIN")

print(
    f"Seizure     : {train_s}"
)

print(
    f"Non-seizure : {train_ns}"
)

print(
    f"Total        : {len(actual_train_df)}"
)


print("\nVALIDATION")

print(
    f"Seizure     : {val_s}"
)

print(
    f"Non-seizure : {val_ns}"
)

print(
    f"Total        : {len(validation_df)}"
)


# ==================================================================================================
# 23. SAVE EXACT SPLITS
# ==================================================================================================

actual_train_df.to_csv(

    TRAIN_SPLIT_CSV,

    index=False
)


validation_df.to_csv(

    VALIDATION_SPLIT_CSV,

    index=False
)


# ==================================================================================================
# 24. LOAD FINAL TEST
# ==================================================================================================

(
    testing_df,
    test_image_column,
    test_label_column

) = load_and_standardize_metadata(

    TESTING_METADATA_CSV,

    "Patient 08 A1 final blinded test",

    "test"
)


# ==================================================================================================
# 25. VERIFY FINAL TEST
# ==================================================================================================

test_s = int(

    (
        testing_df["_label"] == 1
    ).sum()
)


test_ns = int(

    (
        testing_df["_label"] == 0
    ).sum()
)


assert len(testing_df) == EXPECTED_TEST_TOTAL

assert (
    test_s
    + test_ns
    == EXPECTED_TEST_TOTAL
)


print("\nFINAL TEST")

print(
    f"Seizure     : {test_s}"
)

print(
    f"Non-seizure : {test_ns}"
)

print(
    f"Total        : {len(testing_df)}"
)


# ==================================================================================================
# 26. TRAIN / TEST CURRENT PATH LEAKAGE CHECK
# ==================================================================================================

train_test_overlap = set(

    full_training_df[
        "_image_path"
    ]

).intersection(

    set(
        testing_df[
            "_image_path"
        ]
    )

)


if len(
    train_test_overlap
) > 0:

    raise RuntimeError(
        f"\nDATA LEAKAGE DETECTED: "
        f"{len(train_test_overlap)} identical image paths."
    )


print(
    "\nTraining/test current image-path overlap: 0"
)


# ==================================================================================================
# 27. DATASET CLASS
# ==================================================================================================

class EEGTopomapDataset(
    Dataset
):

    def __init__(
        self,
        dataframe
    ):

        self.df = dataframe.reset_index(
            drop=True
        )


    def __len__(
        self
    ):

        return len(
            self.df
        )


    def __getitem__(
        self,
        index
    ):

        row = self.df.iloc[
            index
        ]


        return {

            "index": int(
                index
            ),

            "image_path": str(
                row["_image_path"]
            ),

            "label": int(
                row["_label"]
            )

        }


# ==================================================================================================
# 28. PROCESSOR
# ==================================================================================================

print("\n" + "=" * 100)
print("LOADING MEDGEMMA PROCESSOR")
print("=" * 100)


processor = AutoProcessor.from_pretrained(

    MODEL_ID,

    token=True
)


if hasattr(
    processor,
    "tokenizer"
):

    processor.tokenizer.padding_side = "right"


print(
    "\nProcessor loaded successfully."
)


# ==================================================================================================
# 29. MEDGEMMA LOADER
# ==================================================================================================

def load_medgemma_base():

    try:

        loaded_model = AutoModelForMultimodalLM.from_pretrained(

            MODEL_ID,

            token=True,

            dtype=MODEL_DTYPE,

            low_cpu_mem_usage=True,

            attn_implementation="sdpa"
        )


    except TypeError:

        loaded_model = AutoModelForMultimodalLM.from_pretrained(

            MODEL_ID,

            token=True,

            torch_dtype=MODEL_DTYPE,

            low_cpu_mem_usage=True,

            attn_implementation="sdpa"
        )


    loaded_model = loaded_model.to(
        DEVICE
    )


    return loaded_model


# ==================================================================================================
# 30. LOAD BASE MODEL
# ==================================================================================================

print("\n" + "=" * 100)
print("LOADING MEDGEMMA-4B")
print("=" * 100)


gc.collect()

torch.cuda.empty_cache()


base_model = load_medgemma_base()


print(
    "\nSUCCESS: MedGemma-4B loaded."
)


print(
    f"GPU allocated: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)


# ==================================================================================================
# ==================================================================================================
# 31. MEMORY SETTINGS + FREEZE PRETRAINED MEDGEMMA
# ==================================================================================================

if hasattr(base_model.config, "use_cache"):
    base_model.config.use_cache = False

# Freeze every pretrained MedGemma parameter.
for parameter in base_model.parameters():
    parameter.requires_grad = False

# IMPORTANT FIX:
# LoReFT is injected with a forward hook. Gradient checkpointing can recompute
# transformer blocks during backward and can make custom hook behavior difficult
# to reason about. This corrected run therefore keeps checkpointing OFF.
if hasattr(base_model, "gradient_checkpointing_disable"):
    try:
        base_model.gradient_checkpointing_disable()
    except Exception:
        pass

if hasattr(base_model.config, "gradient_checkpointing"):
    try:
        base_model.config.gradient_checkpointing = False
    except Exception:
        pass

print("Gradient checkpointing disabled.")
print("All pretrained MedGemma parameters frozen.")

if bool(getattr(base_model, "is_gradient_checkpointing", False)):
    raise RuntimeError("Gradient checkpointing is still active; corrected LoReFT run requires it OFF.")


# ==================================================================================================
# 32. LOCATE THE LANGUAGE-MODEL TRANSFORMER STACK
# ==================================================================================================

def find_language_layers(multimodal_model):
    candidates = []

    if hasattr(multimodal_model, "model"):
        outer = multimodal_model.model
        if hasattr(outer, "language_model"):
            candidates.append(("model.language_model", outer.language_model))
        if hasattr(outer, "text_model"):
            candidates.append(("model.text_model", outer.text_model))

    if hasattr(multimodal_model, "language_model"):
        candidates.append(("language_model", multimodal_model.language_model))

    if hasattr(multimodal_model, "text_model"):
        candidates.append(("text_model", multimodal_model.text_model))

    for base_path, language_model in candidates:
        if hasattr(language_model, "layers") and isinstance(language_model.layers, nn.ModuleList):
            return language_model, language_model.layers, f"{base_path}.layers"

        if hasattr(language_model, "model"):
            nested = language_model.model
            if hasattr(nested, "layers") and isinstance(nested.layers, nn.ModuleList):
                return language_model, nested.layers, f"{base_path}.model.layers"

    for module_name, module in multimodal_model.named_modules():
        if (
            isinstance(module, nn.ModuleList)
            and module_name.endswith("layers")
            and "language_model" in module_name.lower()
            and "vision" not in module_name.lower()
        ):
            return None, module, module_name

    raise RuntimeError(
        "Could not locate MedGemma's language transformer stack. "
        "Inspect model.named_modules() before training."
    )


language_model_object, language_layers, LANGUAGE_LAYERS_PATH = find_language_layers(base_model)
NUM_LANGUAGE_LAYERS = len(language_layers)

# Validate and resolve the requested 1-based human layer numbers.
LOREFT_TARGET_LAYERS_HUMAN = [int(x) for x in LOREFT_TARGET_LAYERS_HUMAN]

if len(LOREFT_TARGET_LAYERS_HUMAN) != len(set(LOREFT_TARGET_LAYERS_HUMAN)):
    raise ValueError("Duplicate LoReFT target layers were requested.")

for human_layer in LOREFT_TARGET_LAYERS_HUMAN:
    if human_layer < 1 or human_layer > NUM_LANGUAGE_LAYERS:
        raise ValueError(
            f"Requested LoReFT layer {human_layer}, but the language model has "
            f"{NUM_LANGUAGE_LAYERS} layers."
        )

LOREFT_TARGET_LAYER_INDICES = [
    human_layer - 1
    for human_layer in LOREFT_TARGET_LAYERS_HUMAN
]

if hasattr(base_model.config, "text_config") and hasattr(base_model.config.text_config, "hidden_size"):
    LOREFT_HIDDEN_SIZE = int(base_model.config.text_config.hidden_size)
elif language_model_object is not None and hasattr(language_model_object, "config"):
    LOREFT_HIDDEN_SIZE = int(language_model_object.config.hidden_size)
else:
    raise RuntimeError("Could not determine the language hidden size.")

print("\n" + "=" * 100)
print("MEDGEMMA LANGUAGE STACK")
print("=" * 100)
print(f"Language layer path        : {LANGUAGE_LAYERS_PATH}")
print(f"Number of language layers  : {NUM_LANGUAGE_LAYERS}")
print(f"Language hidden size       : {LOREFT_HIDDEN_SIZE}")
print(f"Intervention human layers  : {LOREFT_TARGET_LAYERS_HUMAN}")
print(f"Intervention Python indices: {LOREFT_TARGET_LAYER_INDICES}")
print(f"Number interventions       : {len(LOREFT_TARGET_LAYER_INDICES)}")


# ==================================================================================================
# 33. LoReFT MODULE
#
# Original LoReFT equation:
#
#       Phi(h) = h + R^T(Wh + b - Rh)
# ==================================================================================================

class LowRankRotateLayer(nn.Module):
    def __init__(self, input_dim, rank):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(input_dim, rank), requires_grad=True)
        nn.init.orthogonal_(self.weight)

    def forward(self, x):
        return torch.matmul(x.to(self.weight.dtype), self.weight)


class LoReFTIntervention(nn.Module):
    def __init__(
        self,
        embed_dim,
        rank,
        train_dtype=torch.float32,
        dropout=0.0
    ):
        super().__init__()

        # ==========================================================================================
        # TRUE FP32 LoReFT PARAMETERS
        # ==========================================================================================
        #
        # The frozen MedGemma backbone runs in BF16, but the trainable LoReFT
        # parameters are deliberately kept in FP32.
        #
        rotate_layer = LowRankRotateLayer(
            embed_dim,
            rank
        ).to(
            dtype=train_dtype
        )

        self.rotate_layer = nn.utils.parametrizations.orthogonal(
            rotate_layer
        )

        self.learned_source = nn.Linear(
            embed_dim,
            rank,
            bias=True
        ).to(
            dtype=train_dtype
        )

        self.dropout = nn.Dropout(
            dropout
        )

        # ==========================================================================================
        # IDENTITY / NO-OP INITIALIZATION
        # ==========================================================================================
        #
        # LoReFT:
        #
        #     Phi(h) = h + R^T(Wh + b - Rh)
        #
        # In this implementation:
        #
        #     rotate_layer(h)  = h @ R
        #
        # while nn.Linear produces:
        #
        #     learned_source(h) = h @ W^T + b
        #
        # Therefore:
        #
        #     W = R^T
        #     b = 0
        #
        # gives:
        #
        #     learned_source(h) = rotate_layer(h)
        #
        # and hence:
        #
        #     delta = 0
        #     Phi(h) = h
        #
        # at initialization.
        #
        # This avoids the very large random representation perturbation that the
        # previous implementation introduced before training.
        #
        with torch.no_grad():

            current_rotation = (
                self.rotate_layer.weight
                .detach()
                .to(
                    dtype=self.learned_source.weight.dtype,
                    device=self.learned_source.weight.device
                )
            )

            self.learned_source.weight.copy_(
                current_rotation.T
            )

            self.learned_source.bias.zero_()

        # Passive diagnostic state.
        self._capture_precision_diagnostic = False
        self._last_precision_diagnostic = None


    def arm_precision_diagnostic(self):

        self._capture_precision_diagnostic = True
        self._last_precision_diagnostic = None


    def get_precision_diagnostic(self):

        return self._last_precision_diagnostic


    def forward(
        self,
        base
    ):

        # ==========================================================================================
        # FORCE LoReFT TO COMPUTE IN TRUE FP32
        # ==========================================================================================
        #
        # The outer MedGemma forward uses BF16 autocast. FP32 parameter storage
        # alone does NOT guarantee FP32 Linear/matmul execution, because autocast
        # may still execute those operations in BF16.
        #
        # We therefore explicitly disable autocast inside the intervention:
        #
        #     BF16 h
        #        -> FP32 h
        #        -> FP32 LoReFT projections
        #        -> FP32 delta
        #        -> FP32 (h + delta)
        #        -> one final cast back to MedGemma's original dtype
        #
        original_dtype = base.dtype

        autocast_device_type = (
            "cuda"
            if base.is_cuda
            else "cpu"
        )

        with torch.autocast(
            device_type=autocast_device_type,
            enabled=False
        ):

            base_fp32 = base.float()

            # Hard safety checks: fail immediately if LoReFT weights are not FP32.
            if self.rotate_layer.weight.dtype != torch.float32:

                raise RuntimeError(
                    "LoReFT rotate_layer weight is not FP32. "
                    f"Found: {self.rotate_layer.weight.dtype}"
                )

            if self.learned_source.weight.dtype != torch.float32:

                raise RuntimeError(
                    "LoReFT learned_source weight is not FP32. "
                    f"Found: {self.learned_source.weight.dtype}"
                )

            rotated_base_fp32 = self.rotate_layer(
                base_fp32
            )

            learned_source_fp32 = self.learned_source(
                base_fp32
            )

            delta_fp32 = torch.matmul(
                learned_source_fp32 - rotated_base_fp32,
                self.rotate_layer.weight.T
            )

            updated_fp32 = (
                base_fp32
                + delta_fp32
            )

        # Only the final updated hidden representation is converted back to BF16.
        updated = updated_fp32.to(
            dtype=original_dtype
        )

        # ==========================================================================================
        # EFFECTIVE INTERVENTION / BF16 DIAGNOSTIC
        # ==========================================================================================
        if self._capture_precision_diagnostic:

            with torch.no_grad():

                base_detached_fp32 = (
                    base
                    .detach()
                    .float()
                )

                raw_delta = (
                    delta_fp32
                    .detach()
                    .float()
                )

                updated_detached_fp32 = (
                    updated
                    .detach()
                    .float()
                )

                effective_delta = (
                    updated_detached_fp32
                    - base_detached_fp32
                )

                raw_norm = float(
                    raw_delta
                    .norm()
                    .cpu()
                )

                effective_norm = float(
                    effective_delta
                    .norm()
                    .cpu()
                )

                cast_error_norm = float(
                    (
                        raw_delta
                        - effective_delta
                    )
                    .norm()
                    .cpu()
                )

                changed_fraction = float(
                    (
                        updated.detach()
                        != base.detach()
                    )
                    .float()
                    .mean()
                    .cpu()
                )

                retained_ratio = (
                    effective_norm / raw_norm
                    if raw_norm > 0.0
                    else 0.0
                )

                self._last_precision_diagnostic = {

                    "base_dtype": str(
                        base.dtype
                    ),

                    "reft_dtype": str(
                        delta_fp32.dtype
                    ),

                    "raw_fp32_delta_norm": raw_norm,

                    "effective_post_cast_delta_norm": effective_norm,

                    "cast_error_norm": cast_error_norm,

                    "retained_norm_ratio": retained_ratio,

                    "changed_element_fraction": changed_fraction,

                    "raw_fp32_delta_mean_abs": float(
                        raw_delta
                        .abs()
                        .mean()
                        .cpu()
                    ),

                    "effective_post_cast_delta_mean_abs": float(
                        effective_delta
                        .abs()
                        .mean()
                        .cpu()
                    ),
                }

            self._capture_precision_diagnostic = False

        return self.dropout(
            updated
        )


# ==================================================================================================
# 34. POSITION CONTROLLER + LANGUAGE-BLOCK HOOK
# ==================================================================================================

class LoReFTPositionController:
    def __init__(self, prefix_tokens, suffix_tokens):
        self.prefix_tokens = int(prefix_tokens)
        self.suffix_tokens = int(suffix_tokens)
        self.active = False
        self.prompt_length = None
        self.minimum_sequence_length = None

    def activate(self, prompt_length, minimum_sequence_length=None):
        self.prompt_length = int(prompt_length)
        self.minimum_sequence_length = int(
            minimum_sequence_length
            if minimum_sequence_length is not None
            else prompt_length
        )
        self.active = True

    def deactivate(self):
        self.active = False
        self.prompt_length = None
        self.minimum_sequence_length = None

    def get_positions(self, current_sequence_length):
        if not self.active:
            return []

        if current_sequence_length < self.minimum_sequence_length:
            # Skip cached one-token decode steps during autoregressive generation.
            return []

        prompt_length = min(self.prompt_length, int(current_sequence_length))

        prefix = list(range(min(self.prefix_tokens, prompt_length)))
        suffix = list(
            range(
                max(0, prompt_length - self.suffix_tokens),
                prompt_length
            )
        )

        positions = []
        seen = set()
        for position in prefix + suffix:
            if position not in seen:
                seen.add(position)
                positions.append(position)

        return positions


def _extract_hidden(layer_output):
    if torch.is_tensor(layer_output):
        return layer_output

    if isinstance(layer_output, (tuple, list)) and len(layer_output) > 0:
        if torch.is_tensor(layer_output[0]):
            return layer_output[0]

    raise TypeError(
        f"Unsupported language-layer output type: {type(layer_output)}"
    )


def _replace_hidden(layer_output, new_hidden):
    if torch.is_tensor(layer_output):
        return new_hidden

    if isinstance(layer_output, tuple):
        return (new_hidden,) + tuple(layer_output[1:])

    if isinstance(layer_output, list):
        return [new_hidden] + list(layer_output[1:])

    raise TypeError(
        f"Unsupported language-layer output type: {type(layer_output)}"
    )


def make_loreft_hook(intervention, controller):
    def hook(module, inputs, output):
        hidden = _extract_hidden(output)

        if hidden.ndim != 3:
            raise RuntimeError(
                f"Expected [batch, sequence, hidden], got {tuple(hidden.shape)}"
            )

        positions = controller.get_positions(hidden.shape[1])
        if not positions:
            return output

        position_tensor = torch.tensor(
            positions,
            dtype=torch.long,
            device=hidden.device
        )

        selected = hidden.index_select(1, position_tensor)
        modified = intervention(selected)

        new_hidden = hidden.clone()
        new_hidden[:, position_tensor, :] = modified

        return _replace_hidden(output, new_hidden)

    return hook


# ==================================================================================================
# 35. ATTACH MULTI-LAYER LoReFT + WRAP MEDGEMMA
# ==================================================================================================

LOREFT_MODULE_NAME = "_loreft_intervention"

# One shared position controller is sufficient because every intervention uses
# the same prompt boundary and the same f5+l5 token positions.
loreft_controller = LoReFTPositionController(
    prefix_tokens=LOREFT_PREFIX_TOKENS,
    suffix_tokens=LOREFT_SUFFIX_TOKENS
)

# Each selected transformer layer gets its OWN independent LoReFT parameters.
target_language_layers = []
loreft_interventions = {}
loreft_hook_handles = []

for human_layer, python_index in zip(
    LOREFT_TARGET_LAYERS_HUMAN,
    LOREFT_TARGET_LAYER_INDICES
):
    target_layer = language_layers[python_index]

    intervention = LoReFTIntervention(
        embed_dim=LOREFT_HIDDEN_SIZE,
        rank=LOREFT_RANK,
        train_dtype=torch.float32,
        dropout=LOREFT_DROPOUT
    ).to(DEVICE)

    target_layer.add_module(
        LOREFT_MODULE_NAME,
        intervention
    )

    hook_handle = target_layer.register_forward_hook(
        make_loreft_hook(
            intervention,
            loreft_controller
        )
    )

    target_language_layers.append(target_layer)
    loreft_interventions[str(human_layer)] = intervention
    loreft_hook_handles.append(hook_handle)


class MedGemmaLoReFTWrapper(nn.Module):
    """
    Thin wrapper around the original multimodal model.

    Multi-layer training forward:
      - derive the prompt boundary from the existing -100 loss mask
      - activate the shared position controller
      - every selected language block applies its own independent LoReFT module
      - gradient checkpointing is disabled

    Generation:
      - intervene at all selected layers during the full prompt-prefill pass
      - skip cached one-token decode passes
    """

    def __init__(
        self,
        base,
        controller,
        interventions,
        checkpoint_info
    ):
        super().__init__()
        self.base_model = base
        self.controller = controller
        self.checkpoint_info = dict(checkpoint_info)

        # The interventions are already registered as children of their target
        # transformer layers. Keep only a non-registered Python reference here.
        object.__setattr__(self, "_interventions_ref", interventions)

    @property
    def config(self):
        return self.base_model.config

    def forward(self, *args, **kwargs):
        labels = kwargs.get("labels", None)

        if labels is not None:
            if labels.ndim != 2 or labels.shape[0] != 1:
                raise ValueError(
                    "This LoReFT pipeline expects physical batch size = 1."
                )

            supervised_positions = torch.nonzero(
                labels[0] != -100,
                as_tuple=False
            ).flatten()

            if supervised_positions.numel() == 0:
                raise RuntimeError(
                    "No assistant target token was found in the loss mask."
                )

            prompt_length = int(supervised_positions[0].item())
        else:
            input_ids = kwargs.get("input_ids", None)
            if input_ids is None:
                raise RuntimeError(
                    "LoReFT forward needs labels or input_ids to determine prompt positions."
                )
            prompt_length = int(input_ids.shape[1])

        self.controller.activate(
            prompt_length=prompt_length,
            minimum_sequence_length=prompt_length
        )

        return self.base_model(*args, **kwargs)

    @torch.no_grad()
    def generate(self, *args, **kwargs):
        input_ids = kwargs.get("input_ids", None)
        if input_ids is None:
            raise RuntimeError("generate() requires input_ids for LoReFT position control.")

        prompt_length = int(input_ids.shape[1])
        self.controller.activate(
            prompt_length=prompt_length,
            minimum_sequence_length=prompt_length
        )

        try:
            return self.base_model.generate(*args, **kwargs)
        finally:
            self.controller.deactivate()

    def save_pretrained(self, save_directory, safe_serialization=True, **kwargs):
        os.makedirs(save_directory, exist_ok=True)

        # Save every intervention in one checkpoint file, keyed by human layer.
        multi_layer_state = {
            layer_key: intervention.state_dict()
            for layer_key, intervention in self._interventions_ref.items()
        }

        torch.save(
            multi_layer_state,
            os.path.join(save_directory, "loreft_intervention.pt")
        )

        with open(
            os.path.join(save_directory, "loreft_config.json"),
            "w",
            encoding="utf-8"
        ) as file:
            json.dump(
                self.checkpoint_info,
                file,
                indent=4
            )


checkpoint_info = {
    "method": "LoReFT",
    "variant": "multi_layer_independent_interventions",
    "formula": "h + R^T(Wh + b - Rh)",
    "rank": LOREFT_RANK,
    "num_language_layers": NUM_LANGUAGE_LAYERS,
    "target_layers_human": LOREFT_TARGET_LAYERS_HUMAN,
    "target_layer_python_indices": LOREFT_TARGET_LAYER_INDICES,
    "num_interventions": len(LOREFT_TARGET_LAYER_INDICES),
    "hidden_size": LOREFT_HIDDEN_SIZE,
    "prefix_prompt_tokens": LOREFT_PREFIX_TOKENS,
    "suffix_prompt_tokens": LOREFT_SUFFIX_TOKENS,
    "dropout": LOREFT_DROPOUT,
    "component": LOREFT_COMPONENT,
    "language_layers_path": LANGUAGE_LAYERS_PATH,
    "precision_fix": "autocast disabled inside LoReFT; true FP32 projection + FP32 residual addition; one final cast to model dtype",
    "initialization": "identity_no_op_W_equals_R_transpose_bias_zero",
    "bf16_diagnostic": True,
    "rotation_weight_decay": 0.0,
    "source_weight_decay": WEIGHT_DECAY,
    "source_bias_weight_decay": 0.0
}

model = MedGemmaLoReFTWrapper(
    base=base_model,
    controller=loreft_controller,
    interventions=loreft_interventions,
    checkpoint_info=checkpoint_info
).to(DEVICE)

trainable_named_parameters = [
    (name, parameter)
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
]

unexpected_trainable = [
    name
    for name, parameter in trainable_named_parameters
    if LOREFT_MODULE_NAME not in name
]

if unexpected_trainable:
    raise RuntimeError(
        "Unexpected pretrained parameters are trainable:\n"
        + "\n".join(unexpected_trainable[:50])
    )

if not trainable_named_parameters:
    raise RuntimeError("No LoReFT parameters are trainable.")

non_fp32_trainable = [
    f"{name}: {parameter.dtype}"
    for name, parameter in trainable_named_parameters
    if parameter.dtype != torch.float32
]

if non_fp32_trainable:
    raise RuntimeError(
        "Every trainable LoReFT parameter must remain FP32, but the following "
        "parameters do not:\n"
        + "\n".join(non_fp32_trainable)
    )

trainable_parameter_count = sum(
    parameter.numel()
    for name, parameter in trainable_named_parameters
)

total_parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_percentage = (
    100.0
    * trainable_parameter_count
    / total_parameter_count
)

expected_per_intervention = (
    2 * LOREFT_HIDDEN_SIZE * LOREFT_RANK
    + LOREFT_RANK
)
expected_total_loreft_parameters = (
    expected_per_intervention
    * len(LOREFT_TARGET_LAYER_INDICES)
)

if trainable_parameter_count != expected_total_loreft_parameters:
    raise RuntimeError(
        "Unexpected multi-layer LoReFT trainable-parameter count.\n"
        f"Expected: {expected_total_loreft_parameters:,}\n"
        f"Found   : {trainable_parameter_count:,}"
    )

print("\n" + "=" * 100)
print("MULTI-LAYER LoReFT TRAINABLE PARAMETER REPORT")
print("=" * 100)

for name, parameter in trainable_named_parameters:
    print(f"  {name}: {parameter.numel():,}")

print(f"\nIntervention layers  : {LOREFT_TARGET_LAYERS_HUMAN}")
print(f"Parameters / layer   : {expected_per_intervention:,}")
print(f"Trainable parameters : {trainable_parameter_count:,}")
print(f"Total parameters     : {total_parameter_count:,}")
print(f"Trainable percentage : {trainable_percentage:.6f}%")
print("LoReFT parameter dtype: torch.float32")
print("LoReFT compute dtype  : torch.float32 (autocast disabled inside intervention)")
print("Initialization        : identity/no-op (W = R^T, b = 0)")

with open(TARGET_MODULES_TXT, "w", encoding="utf-8") as file:
    for key, value in checkpoint_info.items():
        file.write(f"{key}={value}\n")


# ==================================================================================================
# BF16 / FP32 LoReFT PRECISION DIAGNOSTIC HELPERS
# ==================================================================================================

def arm_all_precision_diagnostics(interventions):
    for intervention in interventions.values():
        intervention.arm_precision_diagnostic()


def print_all_precision_diagnostics(interventions, title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    for layer_key in sorted(interventions.keys(), key=lambda x: int(x)):
        stats = interventions[layer_key].get_precision_diagnostic()

        if stats is None:
            print(f"Layer {layer_key}: no diagnostic sample was captured.")
            continue

        print(f"Layer {layer_key}:")
        print(f"  Base hidden dtype             : {stats['base_dtype']}")
        print(f"  LoReFT computation dtype      : {stats['reft_dtype']}")
        print(f"  Raw FP32 delta L2 norm        : {stats['raw_fp32_delta_norm']:.12e}")
        print(f"  Effective post-cast delta norm: {stats['effective_post_cast_delta_norm']:.12e}")
        print(f"  BF16 cast error L2 norm       : {stats['cast_error_norm']:.12e}")
        print(f"  Retained delta-norm ratio     : {stats['retained_norm_ratio']:.6f}")
        print(
            f"  Hidden elements changed       : "
            f"{100.0 * stats['changed_element_fraction']:.4f}%"
        )
        print(f"  Raw FP32 mean |delta|         : {stats['raw_fp32_delta_mean_abs']:.12e}")
        print(f"  Effective mean |delta|        : {stats['effective_post_cast_delta_mean_abs']:.12e}")

    print("=" * 100)


# 36. LABEL -> ASSISTANT TARGET
# ==================================================================================================

def label_to_target_text(
    label
):

    if int(
        label
    ) == 1:

        return "Seizure"


    return "Non-seizure"


# ==================================================================================================
# 37. BUILD PROMPT
# ==================================================================================================

def build_prompt_messages(
    image
):

    return [

        {
            "role": "system",

            "content": [

                {
                    "type": "text",
                    "text": SYSTEM_PROMPT
                }

            ]
        },

        {
            "role": "user",

            "content": [

                {
                    "type": "text",
                    "text": USER_PROMPT
                },

                {
                    "type": "image",
                    "image": image
                }

            ]
        }

    ]


# ==================================================================================================
# 38. BUILD FULL TRAINING CONVERSATION
# ==================================================================================================

def build_training_messages(
    image,
    target_text
):

    messages = build_prompt_messages(
        image
    )


    messages.append(

        {
            "role": "assistant",

            "content": [

                {
                    "type": "text",
                    "text": target_text
                }

            ]
        }

    )


    return messages


# ==================================================================================================
# 39. TRAINING COLLATOR
#
# Prompt/image tokens are masked.
# Loss is calculated only for assistant response.
# ==================================================================================================

class MedGemmaTrainingCollator:

    def __init__(
        self,
        processor
    ):

        self.processor = processor


    def __call__(
        self,
        examples
    ):

        if len(
            examples
        ) != 1:

            raise ValueError(
                "This pipeline expects physical batch size = 1."
            )


        example = examples[
            0
        ]


        image_path = example[
            "image_path"
        ]


        target_text = label_to_target_text(

            example[
                "label"
            ]
        )


        with Image.open(
            image_path
        ) as opened_image:

            image = opened_image.convert(
                "RGB"
            )


            # --------------------------------------------------------------------------------------
            # PROMPT
            # --------------------------------------------------------------------------------------

            prompt_messages = build_prompt_messages(
                image
            )


            prompt_inputs = self.processor.apply_chat_template(

                prompt_messages,

                add_generation_prompt=True,

                tokenize=True,

                return_dict=True,

                return_tensors="pt"
            )


            # --------------------------------------------------------------------------------------
            # FULL TRAINING EXAMPLE
            # --------------------------------------------------------------------------------------

            full_messages = build_training_messages(

                image,

                target_text
            )


            full_inputs = self.processor.apply_chat_template(

                full_messages,

                add_generation_prompt=False,

                tokenize=True,

                return_dict=True,

                return_tensors="pt"
            )


        prompt_ids = prompt_inputs[
            "input_ids"
        ]


        full_ids = full_inputs[
            "input_ids"
        ]


        prompt_length = int(
            prompt_ids.shape[
                1
            ]
        )


        if full_ids.shape[
            1
        ] <= prompt_length:

            raise RuntimeError(
                "Full training conversation contains no target response."
            )


        # ------------------------------------------------------------------------------------------
        # VERIFY EXACT PREFIX
        # ------------------------------------------------------------------------------------------

        prefix_matches = torch.equal(

            full_ids[
                :,
                :prompt_length
            ],

            prompt_ids
        )


        if not prefix_matches:

            raise RuntimeError(
                "\nMedGemma chat template prefix mismatch.\n"
                "Training stopped to avoid an incorrect loss mask."
            )


        # ------------------------------------------------------------------------------------------
        # MASK PROMPT
        # ------------------------------------------------------------------------------------------

        labels = full_ids.clone()


        labels[
            :,
            :prompt_length
        ] = -100


        if int(
            (
                labels != -100
            ).sum()
        ) == 0:

            raise RuntimeError(
                "No assistant target tokens remain after masking."
            )


        full_inputs[
            "labels"
        ] = labels


        return full_inputs


# ==================================================================================================
# 40. DATASETS + LOADERS
# ==================================================================================================

train_dataset = EEGTopomapDataset(
    actual_train_df
)


validation_dataset = EEGTopomapDataset(
    validation_df
)


training_collator = MedGemmaTrainingCollator(
    processor
)


train_generator = torch.Generator()

train_generator.manual_seed(
    RANDOM_SEED
)


train_loader = DataLoader(

    train_dataset,

    batch_size=TRAIN_BATCH_SIZE,

    shuffle=True,

    generator=train_generator,

    num_workers=NUM_WORKERS,

    collate_fn=training_collator
)


validation_loss_loader = DataLoader(

    validation_dataset,

    batch_size=VALIDATION_BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    collate_fn=training_collator
)


# ==================================================================================================
# 41. MOVE MULTIMODAL BATCH TO GPU
# ==================================================================================================

def move_batch_to_device(
    batch
):

    moved = {}


    for key, value in batch.items():

        if not torch.is_tensor(
            value
        ):

            moved[
                key
            ] = value

            continue


        if value.dtype.is_floating_point:

            moved[
                key
            ] = value.to(

                device=DEVICE,

                dtype=MODEL_DTYPE
            )


        else:

            moved[
                key
            ] = value.to(
                DEVICE
            )


    return moved


# ==================================================================================================
# 42. AUTOCAST
# ==================================================================================================

def autocast_context():

    return torch.autocast(

        device_type="cuda",

        dtype=MODEL_DTYPE
    )


# ==================================================================================================
# 43. PRE-FLIGHT TEST
# ==================================================================================================

print("\n" + "=" * 100)
print("LoReFT R4 PRE-FLIGHT TEST")
print("=" * 100)


preview_example = train_dataset[
    0
]


print(
    "\nImage:",
    preview_example[
        "image_path"
    ]
)


print(
    "Target:",
    label_to_target_text(
        preview_example[
            "label"
        ]
    )
)


preview_batch_cpu = training_collator(

    [
        preview_example
    ]
)


target_ids = preview_batch_cpu[
    "labels"
][
    0
]


target_ids = target_ids[
    target_ids != -100
]


print(
    "Assistant target tokens:",
    len(
        target_ids
    )
)


try:

    print(
        "Decoded supervised target:",
        repr(
            processor.decode(
                target_ids,
                skip_special_tokens=False
            )
        )
    )

except Exception:

    pass


preview_batch_gpu = move_batch_to_device(
    preview_batch_cpu
)


model.eval()

# Capture one representative precision diagnostic during the existing pre-flight
# forward. No extra model pass is added.
arm_all_precision_diagnostics(
    loreft_interventions
)


with torch.no_grad():

    with autocast_context():

        preview_output = model(
            **preview_batch_gpu
        )


preview_loss = float(

    preview_output.loss
    .detach()
    .float()
    .cpu()
)


print(
    f"Pre-flight loss: {preview_loss:.6f}"
)

print_all_precision_diagnostics(
    loreft_interventions,
    "PRE-FLIGHT TRUE-FP32 LoReFT INTERVENTION DIAGNOSTIC"
)

print(
    "\nIDENTITY-INITIALIZATION NOTE:\n"
    "At pre-flight, a zero or extremely small LoReFT delta is EXPECTED because "
    "the intervention is intentionally initialized as a no-op (W = R^T, b = 0).\n"
    "After training starts, the important checks are:\n"
    "  1) LoReFT computation dtype = torch.float32\n"
    "  2) validation delta becomes non-zero\n"
    "  3) effective post-cast delta is also non-zero."
)


if not np.isfinite(
    preview_loss
):

    raise RuntimeError(
        "Pre-flight loss is not finite."
    )


print(
    "\nSUCCESS: LoReFT R4 forward pass works."
)


del preview_batch_cpu
del preview_batch_gpu
del preview_output


gc.collect()

torch.cuda.empty_cache()


# ==================================================================================================
# 44. OPTIMIZER
# ==================================================================================================

trainable_parameters = [

    parameter

    for parameter
    in model.parameters()

    if parameter.requires_grad
]


# ==================================================================================================
# IMPORTANT LoReFT OPTIMIZER FIX
# ==================================================================================================
#
# nn.utils.parametrizations.orthogonal() stores a trainable "original" parameter
# underneath the effective orthogonal rotation matrix.
#
# Applying AdamW weight decay directly to that underlying parametrization can
# collapse the EFFECTIVE rotate_layer.weight toward an invalid/zero transform.
# If R collapses to zero, then:
#
#     delta = (Wh + b - Rh) @ R^T
#
# also becomes exactly zero, even though the raw optimizer parameters continue
# to change.
#
# Therefore:
#   - rotation parametrization parameters: weight_decay = 0.0
#   - learned_source.weight: keep the experiment's WEIGHT_DECAY
#   - learned_source.bias: weight_decay = 0.0
#
# This preserves the original regularization intent without destroying the
# orthogonal LoReFT rotation.
# ==================================================================================================

rotation_parameters = []
source_weight_parameters = []
source_bias_parameters = []

for name, parameter in model.named_parameters():

    if not parameter.requires_grad:
        continue

    if "rotate_layer.parametrizations.weight.original" in name:

        rotation_parameters.append(
            parameter
        )

    elif name.endswith(
        "learned_source.bias"
    ):

        source_bias_parameters.append(
            parameter
        )

    elif "learned_source.weight" in name:

        source_weight_parameters.append(
            parameter
        )

    else:

        raise RuntimeError(
            "Unexpected trainable LoReFT parameter in optimizer grouping: "
            f"{name}"
        )


if len(rotation_parameters) != NUM_LOREFT_INTERVENTIONS:

    raise RuntimeError(
        "Expected one trainable orthogonal-rotation parameter per LoReFT layer. "
        f"Expected {NUM_LOREFT_INTERVENTIONS}, found {len(rotation_parameters)}."
    )


optimizer = torch.optim.AdamW(
    [
        {
            "params": rotation_parameters,
            "weight_decay": 0.0,
            "group_name": "loreft_rotation_no_decay",
        },
        {
            "params": source_weight_parameters,
            "weight_decay": WEIGHT_DECAY,
            "group_name": "loreft_source_weight",
        },
        {
            "params": source_bias_parameters,
            "weight_decay": 0.0,
            "group_name": "loreft_source_bias_no_decay",
        },
    ],
    lr=LEARNING_RATE
)


print("\nLoReFT optimizer parameter groups:")
for group in optimizer.param_groups:
    print(
        f"  {group.get('group_name', 'unnamed')}: "
        f"weight_decay={group['weight_decay']}, "
        f"parameters={sum(p.numel() for p in group['params']):,}"
    )


# ==================================================================================================
# LoReFT UPDATE DIAGNOSTICS
# ==================================================================================================

def snapshot_trainable_parameters():
    return [
        parameter.detach().float().cpu().clone()
        for parameter in trainable_parameters
    ]


def effective_rotation_norms():

    norms = {}

    for layer_key, intervention in loreft_interventions.items():

        with torch.no_grad():

            effective_rotation = (
                intervention.rotate_layer.weight
                .detach()
                .float()
            )

            norms[layer_key] = float(
                effective_rotation
                .norm()
                .cpu()
            )

    return norms


def print_effective_rotation_norms(title):

    norms = effective_rotation_norms()

    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

    expected = float(
        LOREFT_RANK ** 0.5
    )

    for layer_key in sorted(
        norms.keys(),
        key=lambda x: int(x)
    ):

        value = norms[layer_key]

        print(
            f"Layer {layer_key}: "
            f"||R||_F = {value:.12e} "
            f"(orthogonal rank-{LOREFT_RANK} expectation ~= {expected:.6f})"
        )

        if (
            not np.isfinite(value)
            or value < 0.5 * expected
        ):

            raise RuntimeError(
                f"Effective LoReFT rotation collapsed at layer {layer_key}. "
                f"Observed ||R||_F={value:.12e}, expected near {expected:.6f}. "
                "Training is stopped."
            )

    print("=" * 100)


def trainable_gradient_l2_norm():
    total_sq = 0.0
    nonzero_tensors = 0
    tensors_with_grad = 0

    for parameter in trainable_parameters:
        if parameter.grad is None:
            continue

        tensors_with_grad += 1
        grad = parameter.grad.detach().float()
        grad_norm_sq = float(torch.sum(grad * grad).cpu())
        total_sq += grad_norm_sq

        if bool(torch.any(grad != 0).item()):
            nonzero_tensors += 1

    return math.sqrt(total_sq), tensors_with_grad, nonzero_tensors


def parameter_change_from_snapshot(before_snapshot):
    total_sq = 0.0
    max_abs = 0.0

    for before, parameter in zip(before_snapshot, trainable_parameters):
        after = parameter.detach().float().cpu()
        delta = after - before
        total_sq += float(torch.sum(delta * delta))
        if delta.numel() > 0:
            max_abs = max(max_abs, float(delta.abs().max()))

    return math.sqrt(total_sq), max_abs


first_nonzero_lr_update_checked = False
zero_lr_warmup_step_reported = False


# ==================================================================================================
# 45. SCHEDULER
# ==================================================================================================

optimizer_steps_per_epoch = math.ceil(

    len(
        train_loader
    )

    /

    GRADIENT_ACCUMULATION_STEPS
)


total_optimizer_steps = (

    optimizer_steps_per_epoch

    *

    NUM_EPOCHS
)


warmup_steps = int(

    total_optimizer_steps

    *

    WARMUP_RATIO
)


scheduler = get_linear_schedule_with_warmup(

    optimizer,

    num_warmup_steps=warmup_steps,

    num_training_steps=total_optimizer_steps
)


# ==================================================================================================
# ==================================================================================================
# 46. PRINT LoReFT R4 SETTINGS
# ==================================================================================================

print("\n" + "=" * 100)
print("LoReFT R4 TRAINING SETTINGS")
print("=" * 100)
print(f"\nEpochs                    : {NUM_EPOCHS}")
print(f"Training images           : {len(train_dataset)}")
print(f"Validation images         : {len(validation_dataset)}")
print(f"Physical batch size       : {TRAIN_BATCH_SIZE}")
print(f"Gradient accumulation     : {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective batch size      : {TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"Learning rate             : {LEARNING_RATE}")
print(f"Warmup ratio              : {WARMUP_RATIO}")
print(f"Warmup optimizer steps    : {warmup_steps}")
print(f"Initial scheduled LR      : {optimizer.param_groups[0]['lr']:.12e}")
print(f"LoReFT rank               : {LOREFT_RANK}")
print(f"Language layers           : {NUM_LANGUAGE_LAYERS}")
print(f"Intervention human layers : {LOREFT_TARGET_LAYERS_HUMAN}")
print(f"Intervention Python idx   : {LOREFT_TARGET_LAYER_INDICES}")
print(f"Number interventions      : {len(LOREFT_TARGET_LAYER_INDICES)}")
print(f"Prefix prompt tokens      : {LOREFT_PREFIX_TOKENS}")
print(f"Suffix prompt tokens      : {LOREFT_SUFFIX_TOKENS}")
print(f"LoReFT dropout            : {LOREFT_DROPOUT}")
print(f"Component                 : {LOREFT_COMPONENT}")
print("Base MedGemma weights     : FROZEN")
print("Vision tower              : FROZEN")
print("Gradient checkpointing    : OFF")
print("LoReFT parameter dtype    : float32")
print("LoReFT compute dtype      : float32 (autocast disabled inside LoReFT)")
print("LoReFT initialization     : identity/no-op (W = R^T, b = 0)")
print("Rotation weight decay     : 0.0  <-- IMPORTANT orthogonal-parametrization fix")
print(f"Source weight decay       : {WEIGHT_DECAY}")
print("Source bias weight decay  : 0.0")
print("Checkpoint deletion       : DISABLED (new folder per best epoch)")
print(f"Run ID                    : {RUN_ID}")
print(f"Output folder             : {OUTPUT_ROOT}")

# 47. GRAD SCALER
# ==================================================================================================

if MODEL_DTYPE == torch.float16:

    try:

        grad_scaler = torch.amp.GradScaler(
            "cuda"
        )

    except Exception:

        grad_scaler = torch.cuda.amp.GradScaler()


else:

    grad_scaler = None


print_effective_rotation_norms(
    "PRE-TRAIN EFFECTIVE LoReFT ROTATION-NORM CHECK"
)


# ==================================================================================================
# 48. VALIDATION LOSS
# ==================================================================================================

@torch.no_grad()
def calculate_validation_loss():

    model.eval()


    losses = []


    progress = tqdm(

        validation_loss_loader,

        desc="Validation loss",

        leave=False
    )


    for batch in progress:

        batch = move_batch_to_device(
            batch
        )


        with autocast_context():

            outputs = model(
                **batch
            )


        loss = outputs.loss


        loss_value = float(

            loss
            .detach()
            .float()
            .cpu()
        )


        losses.append(
            loss_value
        )


        progress.set_postfix(
            loss=f"{loss_value:.4f}"
        )


        del batch
        del outputs
        del loss


    return float(
        np.mean(
            losses
        )
    )


# ==================================================================================================
# 49. NORMALIZE MODEL OUTPUT
# ==================================================================================================

def normalize_prediction_text(
    output_text
):

    text = str(
        output_text
    ).strip().lower()


    text = text.replace(
        "_",
        " "
    )


    text = text.replace(
        "–",
        "-"
    )


    text = re.sub(
        r"\s+",
        " ",
        text
    )


    # NON-SEIZURE FIRST

    if re.search(
        r"\bnon[\s-]*seizure\b",
        text
    ):

        return 0


    if re.search(
        r"\bseizure\b",
        text
    ):

        return 1


    return None


# ==================================================================================================
# 50. SINGLE IMAGE INFERENCE
# ==================================================================================================

@torch.no_grad()
def predict_single_image(
    image_path
):

    model.eval()


    with Image.open(
        image_path
    ) as opened_image:

        image = opened_image.convert(
            "RGB"
        )


        messages = build_prompt_messages(
            image
        )


        inputs = processor.apply_chat_template(

            messages,

            add_generation_prompt=True,

            tokenize=True,

            return_dict=True,

            return_tensors="pt"
        )


    input_length = int(

        inputs[
            "input_ids"
        ].shape[
            1
        ]
    )


    inputs = move_batch_to_device(
        inputs
    )


    with autocast_context():

        generated = model.generate(

            **inputs,

            max_new_tokens=MAX_NEW_TOKENS,

            do_sample=False,

            num_beams=1,

            use_cache=True
        )


    generated_tokens = generated[

        0,

        input_length:

    ]


    decoded = processor.decode(

        generated_tokens,

        skip_special_tokens=True
    ).strip()


    prediction = normalize_prediction_text(
        decoded
    )


    return (
        decoded,
        prediction
    )


# ==================================================================================================
# 51. METRIC FUNCTION
# ==================================================================================================

def calculate_binary_metrics(
    true_labels,
    predictions
):

    metric_predictions = []

    invalid_count = 0


    for truth, prediction in zip(

        true_labels,

        predictions

    ):

        if prediction is None:

            invalid_count += 1


            # Invalid generation is counted as WRONG.

            metric_predictions.append(

                1
                -
                int(
                    truth
                )
            )


        else:

            metric_predictions.append(
                int(
                    prediction
                )
            )


    y_true = np.asarray(
        true_labels,
        dtype=int
    )


    y_pred = np.asarray(
        metric_predictions,
        dtype=int
    )


    cm = confusion_matrix(

        y_true,

        y_pred,

        labels=[
            0,
            1
        ]
    )


    tn, fp, fn, tp = cm.ravel()


    sensitivity = (

        tp
        /
        (
            tp
            +
            fn
        )

        if
        (
            tp
            +
            fn
        ) > 0

        else
        0.0
    )


    specificity = (

        tn
        /
        (
            tn
            +
            fp
        )

        if
        (
            tn
            +
            fp
        ) > 0

        else
        0.0
    )


    metrics = {

        "TN": int(
            tn
        ),

        "FP": int(
            fp
        ),

        "FN": int(
            fn
        ),

        "TP": int(
            tp
        ),

        "Sensitivity": float(
            sensitivity
        ),

        "Specificity": float(
            specificity
        ),

        "Balanced_Accuracy": float(

            balanced_accuracy_score(
                y_true,
                y_pred
            )
        ),

        "Accuracy": float(

            accuracy_score(
                y_true,
                y_pred
            )
        ),

        "Precision": float(

            precision_score(
                y_true,
                y_pred,
                zero_division=0
            )
        ),

        "F1": float(

            f1_score(
                y_true,
                y_pred,
                zero_division=0
            )
        ),

        "Invalid_Predictions": int(
            invalid_count
        )

    }


    return (
        metrics,
        y_pred
    )


# ==================================================================================================
# 52. VALIDATION GENERATION
# ==================================================================================================

@torch.no_grad()
def evaluate_validation_generation(
    epoch
):

    model.eval()


    true_labels = []

    predictions = []

    rows = []


    progress = tqdm(

        range(
            len(
                validation_df
            )
        ),

        desc=f"Validation predictions epoch {epoch}"
    )


    for index in progress:

        row = validation_df.iloc[
            index
        ]


        image_path = str(
            row["_image_path"]
        )


        # MODEL PREDICTS BEFORE GROUND TRUTH IS USED.

        raw_output, prediction = predict_single_image(
            image_path
        )


        true_label = int(
            row["_label"]
        )


        true_labels.append(
            true_label
        )


        predictions.append(
            prediction
        )


        rows.append(

            {

                "index": index,

                "image_path": image_path,

                "true_label": true_label,

                "true_class": (
                    "Seizure"
                    if
                    true_label == 1
                    else
                    "Non-seizure"
                ),

                "raw_model_output": raw_output,

                "prediction": (
                    prediction
                    if
                    prediction is not None
                    else
                    "INVALID"
                ),

                "predicted_class": (
                    "Seizure"
                    if
                    prediction == 1
                    else
                    "Non-seizure"
                    if
                    prediction == 0
                    else
                    "INVALID"
                )

            }

        )


    metrics, metric_predictions = calculate_binary_metrics(

        true_labels,

        predictions
    )


    validation_prediction_df = pd.DataFrame(
        rows
    )


    validation_prediction_df[
        "metric_prediction"
    ] = metric_predictions


    validation_prediction_df[
        "correct"
    ] = (

        validation_prediction_df[
            "true_label"
        ].to_numpy()

        ==

        metric_predictions
    )


    validation_output_csv = os.path.join(

        OUTPUT_ROOT,

        f"R4_validation_predictions_epoch_{epoch}.csv"
    )


    validation_prediction_df.to_csv(

        validation_output_csv,

        index=False
    )


    return metrics


# ==================================================================================================
# ==================================================================================================
# 53. SAVE EXPERIMENT CONFIGURATION
# ==================================================================================================

experiment_config = {
    "experiment": "R4",
    "patient": 24,
    "representation": "A1-coefficient fixed-scale EEG variance topomap",
    "model": MODEL_ID,
    "fine_tuning_method": "LoReFT",
    "loreft_formula": "h + R^T(Wh + b - Rh)",
    "quantization": "None",
    "random_seed": RANDOM_SEED,
    "system_prompt": SYSTEM_PROMPT,
    "user_prompt": USER_PROMPT,
    "training_images": len(actual_train_df),
    "validation_images": len(validation_df),
    "testing_images": len(testing_df),
    "loreft_rank": LOREFT_RANK,
    "num_language_layers": NUM_LANGUAGE_LAYERS,
    "loreft_target_layers_human": LOREFT_TARGET_LAYERS_HUMAN,
    "loreft_target_layer_python_indices": LOREFT_TARGET_LAYER_INDICES,
    "loreft_num_interventions": len(LOREFT_TARGET_LAYER_INDICES),
    "language_layers_path": LANGUAGE_LAYERS_PATH,
    "hidden_size": LOREFT_HIDDEN_SIZE,
    "prefix_prompt_tokens": LOREFT_PREFIX_TOKENS,
    "suffix_prompt_tokens": LOREFT_SUFFIX_TOKENS,
    "loreft_dropout": LOREFT_DROPOUT,
    "component": LOREFT_COMPONENT,
    "gradient_checkpointing": False,
    "loreft_train_dtype": "float32",
    "checkpoint_save_strategy": "new directory per best epoch; no rmtree",
    "epochs": NUM_EPOCHS,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "best_model_primary_criterion": "Highest validation balanced accuracy",
    "best_model_tie_breaker": "Lower validation loss",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "physical_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    "trainable_parameters": trainable_parameter_count,
    "total_parameters": total_parameter_count,
    "trainable_percentage": trainable_percentage,
    "precision": PRECISION_NAME,
    "gpu": GPU_NAME,
    "gpu_memory_gb": GPU_MEMORY_GB,
    "pytorch_version": torch.__version__,
    "transformers_version": transformers.__version__
}

with open(EXPERIMENT_CONFIG_JSON, "w", encoding="utf-8") as file:
    json.dump(experiment_config, file, indent=4)

# 54. CHECKPOINT STORAGE (NO DIRECTORY DELETION)
# ==================================================================================================
# OneDrive can temporarily lock folders and cause WinError 5 when shutil.rmtree()
# is used. This corrected run NEVER deletes checkpoint directories. Each new best
# epoch is written to its own fresh folder under BEST_CHECKPOINT_ROOT.
print("\nCheckpoint strategy: new folder per best epoch; no shutil.rmtree().")
print("Checkpoint root    :", BEST_CHECKPOINT_ROOT)


# ==================================================================================================
# 55. TRAINING VARIABLES
# ==================================================================================================

training_history = []


best_validation_balanced_accuracy = -float(
    "inf"
)


best_validation_loss = float(
    "inf"
)


best_epoch = None


# Early stopping counts consecutive epochs that fail to create a new best checkpoint.
epochs_without_improvement = 0

stopped_early = False

early_stop_epoch = None


optimizer.zero_grad(
    set_to_none=True
)


torch.cuda.reset_peak_memory_stats(
    DEVICE
)


training_start_time = time.time()


# ==================================================================================================
# 56. START R4 TRAINING
# ==================================================================================================

print("\n" + "=" * 100)
print("STARTING PATIENT 08 A1 MEDGEMMA-4B LoReFT R4 TRAINING")
print("=" * 100)


for epoch in range(

    1,

    NUM_EPOCHS + 1

):

    print("\n" + "=" * 100)

    print(
        f"EPOCH {epoch}/{NUM_EPOCHS}"
    )

    print("=" * 100)


    model.train()


    epoch_losses = []


    epoch_start_time = time.time()

    # Snapshot LoReFT parameters so we can verify they changed during this epoch.
    epoch_parameter_snapshot = snapshot_trainable_parameters()


    optimizer.zero_grad(
        set_to_none=True
    )


    progress = tqdm(

        enumerate(
            train_loader,
            start=1
        ),

        total=len(
            train_loader
        ),

        desc=f"R4 training epoch {epoch}"
    )


    accumulation_count = 0


    for batch_number, batch in progress:

        try:

            batch = move_batch_to_device(
                batch
            )


            with autocast_context():

                outputs = model(
                    **batch
                )


                raw_loss = outputs.loss


                scaled_loss = (

                    raw_loss

                    /

                    GRADIENT_ACCUMULATION_STEPS
                )


            if not torch.isfinite(
                raw_loss
            ):

                raise RuntimeError(
                    "Training loss became NaN/Inf."
                )


            if grad_scaler is not None:

                grad_scaler.scale(
                    scaled_loss
                ).backward()


            else:

                scaled_loss.backward()


            accumulation_count += 1


            current_loss = float(

                raw_loss
                .detach()
                .float()
                .cpu()
            )


            epoch_losses.append(
                current_loss
            )


            should_update = (

                accumulation_count
                ==
                GRADIENT_ACCUMULATION_STEPS

                or

                batch_number
                ==
                len(
                    train_loader
                )
            )


            if should_update:

                if grad_scaler is not None:
                    grad_scaler.unscale_(optimizer)

                gradient_l2, grad_tensor_count, nonzero_grad_tensors = trainable_gradient_l2_norm()

                # IMPORTANT:
                # Hugging Face's linear warmup scheduler initializes the optimizer LR at 0.
                # Therefore the VERY FIRST optimizer.step() is expected to make no parameter
                # change. We must verify learning on the first optimizer step whose LR is > 0,
                # not blindly on optimizer step #1.
                optimizer_lr_before_step = float(optimizer.param_groups[0]["lr"])
                diagnostic_this_step = False

                if (
                    not first_nonzero_lr_update_checked
                    and optimizer_lr_before_step > 0.0
                ):
                    diagnostic_this_step = True
                    first_update_before = snapshot_trainable_parameters()

                    print("\n" + "=" * 100)
                    print("FIRST NON-ZERO-LR LoReFT UPDATE DIAGNOSTIC")
                    print("=" * 100)
                    print(f"Optimizer LR before step     : {optimizer_lr_before_step:.12e}")
                    print(f"Gradient L2 norm before clip : {gradient_l2:.12e}")
                    print(f"Gradient tensors present     : {grad_tensor_count}")
                    print(f"Gradient tensors non-zero    : {nonzero_grad_tensors}")

                    if grad_tensor_count == 0 or nonzero_grad_tensors == 0 or gradient_l2 <= 0.0:
                        raise RuntimeError(
                            "LoReFT received no non-zero gradient on the first non-zero-LR optimizer update. "
                            "Training is stopped instead of silently continuing."
                        )

                elif (
                    not first_nonzero_lr_update_checked
                    and optimizer_lr_before_step == 0.0
                    and not zero_lr_warmup_step_reported
                ):
                    print("\n" + "=" * 100)
                    print("LoReFT WARMUP NOTE")
                    print("=" * 100)
                    print("Optimizer LR before step     : 0.000000000000e+00")
                    print(f"Gradient L2 norm before clip : {gradient_l2:.12e}")
                    print(f"Gradient tensors present     : {grad_tensor_count}")
                    print(f"Gradient tensors non-zero    : {nonzero_grad_tensors}")
                    print(
                        "The first optimizer step uses LR=0 because of the configured linear warmup. "
                        "A zero parameter change on this step is EXPECTED."
                    )
                    print(
                        "The learning diagnostic is deferred to the first optimizer step with LR > 0."
                    )
                    print("=" * 100)
                    zero_lr_warmup_step_reported = True

                clipped_norm = torch.nn.utils.clip_grad_norm_(
                    trainable_parameters,
                    MAX_GRAD_NORM
                )

                if grad_scaler is not None:
                    grad_scaler.step(optimizer)
                    grad_scaler.update()
                else:
                    optimizer.step()

                # Standard order: optimizer.step() first, then scheduler.step().
                scheduler.step()

                if diagnostic_this_step:
                    first_delta_l2, first_delta_max = parameter_change_from_snapshot(first_update_before)
                    print(f"Gradient norm returned by clip: {float(clipped_norm):.12e}")
                    print(f"Parameter delta L2           : {first_delta_l2:.12e}")
                    print(f"Parameter delta max abs      : {first_delta_max:.12e}")

                    if first_delta_l2 <= 0.0 or first_delta_max <= 0.0:
                        raise RuntimeError(
                            "LoReFT parameters did not change on the first optimizer step with LR > 0. "
                            "Training is stopped because the intervention is not learning."
                        )

                    print("SUCCESS: LoReFT parameters changed on the first non-zero-LR optimizer step.")
                    print("=" * 100)

                    print_effective_rotation_norms(
                        "POST-FIRST-NONZERO-LR EFFECTIVE ROTATION-NORM CHECK"
                    )

                    first_nonzero_lr_update_checked = True

                optimizer.zero_grad(set_to_none=True)
                accumulation_count = 0


            current_lr = scheduler.get_last_lr()[
                0
            ]


            gpu_gb = (

                torch.cuda.memory_allocated()

                /

                1024 ** 3
            )


            progress.set_postfix(

                loss=f"{current_loss:.4f}",

                lr=f"{current_lr:.2e}",

                gpu=f"{gpu_gb:.2f}GB"
            )


            del batch
            del outputs
            del raw_loss
            del scaled_loss


        except torch.cuda.OutOfMemoryError as error:

            torch.cuda.empty_cache()


            raise RuntimeError(
                "\nCUDA OUT OF MEMORY DURING R4.\n"
                "Do NOT switch this run to 4-bit because that would "
                "turn the experiment into QLoRA."
            ) from error


    # ==================================================================================================
    # EPOCH TRAIN LOSS
    # ==================================================================================================

    mean_training_loss = float(

        np.mean(
            epoch_losses
        )
    )


    epoch_parameter_delta_l2, epoch_parameter_delta_max = parameter_change_from_snapshot(
        epoch_parameter_snapshot
    )

    print("\nLoReFT epoch parameter change:")
    print(f"  L2 delta      : {epoch_parameter_delta_l2:.12e}")
    print(f"  Max abs delta : {epoch_parameter_delta_max:.12e}")

    if epoch_parameter_delta_l2 <= 0.0 or epoch_parameter_delta_max <= 0.0:
        raise RuntimeError(
            f"LoReFT parameters did not change during epoch {epoch}. "
            "Training is stopped because the intervention is not learning."
        )

    print_effective_rotation_norms(
        f"EPOCH {epoch} EFFECTIVE LoReFT ROTATION-NORM CHECK"
    )


    # ==================================================================================================
    # VALIDATION LOSS
    # ==================================================================================================

    print(
        "\nCalculating validation loss..."
    )

    # Capture the first normal validation forward of this epoch.
    # This is passive diagnostics only; it does not add another forward pass.
    arm_all_precision_diagnostics(
        loreft_interventions
    )

    mean_validation_loss = calculate_validation_loss()

    print_all_precision_diagnostics(
        loreft_interventions,
        f"EPOCH {epoch} FIRST-VALIDATION-BATCH TRUE-FP32 LoReFT DIAGNOSTIC"
    )


    # ==================================================================================================
    # VALIDATION CLASSIFICATION
    # ==================================================================================================

    print(
        "\nRunning validation classification..."
    )


    validation_metrics = evaluate_validation_generation(
        epoch
    )


    validation_ba = validation_metrics[
        "Balanced_Accuracy"
    ]


    epoch_minutes = (

        time.time()

        -

        epoch_start_time

    ) / 60.0


    # ==================================================================================================
    # PRINT RESULTS
    # ==================================================================================================

    print("\n" + "-" * 100)

    print(
        f"R4 EPOCH {epoch} RESULTS"
    )

    print("-" * 100)


    print(
        f"Training loss       : {mean_training_loss:.6f}"
    )

    print(
        f"Validation loss     : {mean_validation_loss:.6f}"
    )


    print(
        f"TN                  : {validation_metrics['TN']}"
    )

    print(
        f"FP                  : {validation_metrics['FP']}"
    )

    print(
        f"FN                  : {validation_metrics['FN']}"
    )

    print(
        f"TP                  : {validation_metrics['TP']}"
    )


    print(
        f"Sensitivity         : "
        f"{validation_metrics['Sensitivity']:.4f}"
    )

    print(
        f"Specificity         : "
        f"{validation_metrics['Specificity']:.4f}"
    )

    print(
        f"Balanced Accuracy   : "
        f"{validation_metrics['Balanced_Accuracy']:.4f}"
    )

    print(
        f"Accuracy            : "
        f"{validation_metrics['Accuracy']:.4f}"
    )

    print(
        f"F1                  : "
        f"{validation_metrics['F1']:.4f}"
    )


    # ==================================================================================================
    # HISTORY
    # ==================================================================================================

    training_history.append(

        {

            "epoch": epoch,

            "training_loss": mean_training_loss,

            "validation_loss": mean_validation_loss,

            "TN": validation_metrics[
                "TN"
            ],

            "FP": validation_metrics[
                "FP"
            ],

            "FN": validation_metrics[
                "FN"
            ],

            "TP": validation_metrics[
                "TP"
            ],

            "sensitivity": validation_metrics[
                "Sensitivity"
            ],

            "specificity": validation_metrics[
                "Specificity"
            ],

            "balanced_accuracy": validation_metrics[
                "Balanced_Accuracy"
            ],

            "accuracy": validation_metrics[
                "Accuracy"
            ],

            "precision": validation_metrics[
                "Precision"
            ],

            "f1": validation_metrics[
                "F1"
            ],

            "invalid_predictions": validation_metrics[
                "Invalid_Predictions"
            ],

            "epoch_minutes": epoch_minutes,
            "loreft_parameter_delta_l2": epoch_parameter_delta_l2,
            "loreft_parameter_delta_max_abs": epoch_parameter_delta_max

        }

    )


    pd.DataFrame(
        training_history
    ).to_csv(

        TRAINING_HISTORY_CSV,

        index=False
    )


    # ==================================================================================================
    # BEST CHECKPOINT SELECTION
    #
    # Primary: Highest validation BA
    # Tie: Lower validation loss
    # ==================================================================================================

    is_better = False


    if (
        validation_ba
        >
        best_validation_balanced_accuracy
    ):

        is_better = True


    elif (

        math.isclose(

            validation_ba,

            best_validation_balanced_accuracy,

            rel_tol=0.0,

            abs_tol=1e-12
        )

        and

        mean_validation_loss
        <
        best_validation_loss

    ):

        is_better = True


    if is_better:

        best_validation_balanced_accuracy = validation_ba

        best_validation_loss = mean_validation_loss

        best_epoch = epoch


        # IMPORTANT: Never delete an old OneDrive checkpoint folder.
        # Every new best epoch gets a fresh directory, so WinError 5 from rmtree
        # cannot occur here.
        BEST_ADAPTER_DIR = os.path.join(
            BEST_CHECKPOINT_ROOT,
            f"e{epoch:03d}"
        )
        os.makedirs(BEST_ADAPTER_DIR, exist_ok=False)

        BEST_REFT_WEIGHTS = os.path.join(BEST_ADAPTER_DIR, "loreft_intervention.pt")
        BEST_REFT_CONFIG = os.path.join(BEST_ADAPTER_DIR, "loreft_config.json")

        model.save_pretrained(
            BEST_ADAPTER_DIR,
            safe_serialization=True
        )

        # The processor is unchanged from google/medgemma-4b-it, so it does not
        # need to be copied into every best-checkpoint folder.


        print("\n" + "*" * 100)

        print(
            "NEW BEST R4 LoReFT CHECKPOINT SAVED"
        )

        print("*" * 100)


        print(
            f"Best epoch        : {best_epoch}"
        )

        print(
            f"Validation BA     : "
            f"{best_validation_balanced_accuracy:.4f}"
        )

        print(
            f"Validation loss   : "
            f"{best_validation_loss:.6f}"
        )


        # A new best checkpoint resets the early-stopping patience counter.
        epochs_without_improvement = 0

        print(
            f"Early stopping    : 0/{EARLY_STOPPING_PATIENCE}"
        )


    else:

        # No new best checkpoint this epoch.
        epochs_without_improvement += 1

        print("\n" + "-" * 100)

        print(
            f"NO NEW BEST CHECKPOINT | "
            f"Early stopping patience: "
            f"{epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}"
        )

        print("-" * 100)


    gc.collect()

    torch.cuda.empty_cache()


    # ==================================================================================================
    # EARLY STOPPING
    #
    # Stop only after 7 CONSECUTIVE epochs fail to create a new best checkpoint.
    # The best saved checkpoint is kept and will be reloaded for the final blinded test.
    # ==================================================================================================

    if (
        epochs_without_improvement
        >=
        EARLY_STOPPING_PATIENCE
    ):

        stopped_early = True

        early_stop_epoch = epoch

        print("\n" + "=" * 100)
        print("EARLY STOPPING TRIGGERED")
        print("=" * 100)

        print(
            f"\nNo new best checkpoint for "
            f"{EARLY_STOPPING_PATIENCE} consecutive epochs."
        )

        print(
            f"Stopping at epoch     : {early_stop_epoch}"
        )

        print(
            f"Best epoch            : {best_epoch}"
        )

        print(
            f"Best validation BA    : "
            f"{best_validation_balanced_accuracy:.4f}"
        )

        print(
            f"Best validation loss  : "
            f"{best_validation_loss:.6f}"
        )

        print(
            "The final blinded test will use the BEST saved checkpoint, "
            "not the stopping epoch."
        )

        break


# ==================================================================================================
# 57. TRAINING COMPLETE
# ==================================================================================================

total_training_minutes = (

    time.time()

    -

    training_start_time

) / 60.0


peak_training_memory_gb = (

    torch.cuda.max_memory_allocated(
        DEVICE
    )

    /

    1024 ** 3
)


print("\n" + "=" * 100)
print("R4 TRAINING COMPLETE")
print("=" * 100)


print(
    f"\nBest epoch         : {best_epoch}"
)

print(
    f"Best validation BA : "
    f"{best_validation_balanced_accuracy:.4f}"
)

print(
    f"Best validation loss: "
    f"{best_validation_loss:.6f}"
)

print(
    f"Early stop patience : {EARLY_STOPPING_PATIENCE}"
)

print(
    f"Stopped early       : {stopped_early}"
)

if stopped_early:

    print(
        f"Stopped at epoch    : {early_stop_epoch}"
    )

print(
    f"Training time      : "
    f"{total_training_minutes:.2f} minutes"
)

print(
    f"Peak GPU allocated : "
    f"{peak_training_memory_gb:.2f} GB"
)


if best_epoch is None or BEST_ADAPTER_DIR is None:
    raise RuntimeError("No best checkpoint was saved.")

BEST_REFT_WEIGHTS = os.path.join(BEST_ADAPTER_DIR, "loreft_intervention.pt")
BEST_REFT_CONFIG = os.path.join(BEST_ADAPTER_DIR, "loreft_config.json")

if not os.path.isfile(BEST_REFT_WEIGHTS):
    raise FileNotFoundError(f"Best LoReFT weights were not found: {BEST_REFT_WEIGHTS}")
if not os.path.isfile(BEST_REFT_CONFIG):
    raise FileNotFoundError(f"Best LoReFT config was not found: {BEST_REFT_CONFIG}")


# ==================================================================================================
# 58. DELETE TRAINING MODEL + RELEASE TRAINING HOOK REFERENCES
# ==================================================================================================

for hook_handle in loreft_hook_handles:
    try:
        hook_handle.remove()
    except Exception:
        pass

del model
del base_model
del optimizer
del scheduler
del trainable_parameters

# These objects otherwise keep references to the old transformer stack.
del language_layers
del language_model_object
del target_language_layers
del loreft_hook_handles
del loreft_interventions
del loreft_controller

gc.collect()
torch.cuda.empty_cache()


# ==================================================================================================
# 59. LOAD FRESH MEDGEMMA
# ==================================================================================================

print("\n" + "=" * 100)
print("LOADING FRESH MEDGEMMA FOR FINAL LoReFT R4 TEST")
print("=" * 100)


base_model_for_test = load_medgemma_base()


if hasattr(
    base_model_for_test.config,
    "use_cache"
):

    base_model_for_test.config.use_cache = True


# ==================================================================================================
# ==================================================================================================
# 60. LOAD BEST LoReFT R4 INTERVENTION
# ==================================================================================================

with open(BEST_REFT_CONFIG, "r", encoding="utf-8") as file:
    best_reft_config = json.load(file)

for parameter in base_model_for_test.parameters():
    parameter.requires_grad = False

_, test_language_layers, test_language_layers_path = find_language_layers(
    base_model_for_test
)

if len(test_language_layers) != int(best_reft_config["num_language_layers"]):
    raise RuntimeError(
        "Language-layer count differs between training and final test."
    )

test_target_human_layers = [
    int(x)
    for x in best_reft_config["target_layers_human"]
]
test_target_indices = [
    int(x)
    for x in best_reft_config["target_layer_python_indices"]
]

if len(test_target_human_layers) != len(test_target_indices):
    raise RuntimeError("Saved LoReFT target-layer metadata is inconsistent.")

try:
    saved_reft_state = torch.load(
        BEST_REFT_WEIGHTS,
        map_location=DEVICE,
        weights_only=True
    )
except TypeError:
    saved_reft_state = torch.load(
        BEST_REFT_WEIGHTS,
        map_location=DEVICE
    )

if not isinstance(saved_reft_state, dict):
    raise RuntimeError("Saved multi-layer LoReFT state is not a dictionary.")

test_loreft_controller = LoReFTPositionController(
    prefix_tokens=int(best_reft_config["prefix_prompt_tokens"]),
    suffix_tokens=int(best_reft_config["suffix_prompt_tokens"])
)

test_target_layers = []
test_loreft_interventions = {}
test_loreft_hook_handles = []

for human_layer, python_index in zip(
    test_target_human_layers,
    test_target_indices
):
    layer_key = str(human_layer)

    if layer_key not in saved_reft_state:
        raise RuntimeError(
            f"Missing saved LoReFT state for human layer {human_layer}."
        )

    target_layer = test_language_layers[python_index]

    intervention = LoReFTIntervention(
        embed_dim=int(best_reft_config["hidden_size"]),
        rank=int(best_reft_config["rank"]),
        train_dtype=torch.float32,
        dropout=float(best_reft_config["dropout"])
    ).to(DEVICE)

    target_layer.add_module(
        LOREFT_MODULE_NAME,
        intervention
    )

    intervention.load_state_dict(
        saved_reft_state[layer_key],
        strict=True
    )

    for parameter in intervention.parameters():
        parameter.requires_grad = False

    hook_handle = target_layer.register_forward_hook(
        make_loreft_hook(
            intervention,
            test_loreft_controller
        )
    )

    test_target_layers.append(target_layer)
    test_loreft_interventions[layer_key] = intervention
    test_loreft_hook_handles.append(hook_handle)

model = MedGemmaLoReFTWrapper(
    base=base_model_for_test,
    controller=test_loreft_controller,
    interventions=test_loreft_interventions,
    checkpoint_info=best_reft_config
).to(DEVICE)

model.eval()

print(
    f"\nBest LoReFT R4 checkpoint loaded from epoch {best_epoch}."
)
print(
    f"Target layers: human {best_reft_config['target_layers_human']} / "
    f"{best_reft_config['num_language_layers']}"
)

# 61. FINAL 1000-IMAGE BLINDED TEST
# ==================================================================================================

print("\n" + "=" * 100)
print("STARTING LoReFT R4 FINAL BLINDED TEST")
print("=" * 100)


test_rows = []

test_true_labels = []

test_predictions = []


test_start_time = time.time()


progress = tqdm(

    range(
        len(
            testing_df
        )
    ),

    desc="LoReFT R4 final blinded test"
)


for index in progress:

    row = testing_df.iloc[
        index
    ]


    image_path = str(
        row["_image_path"]
    )


    # ----------------------------------------------------------------------------------------------
    # MODEL PREDICTION FIRST
    # ----------------------------------------------------------------------------------------------

    raw_output, prediction = predict_single_image(
        image_path
    )


    # ----------------------------------------------------------------------------------------------
    # GROUND TRUTH READ AFTER GENERATION
    # ----------------------------------------------------------------------------------------------

    true_label = int(
        row["_label"]
    )


    test_true_labels.append(
        true_label
    )


    test_predictions.append(
        prediction
    )


    test_rows.append(

        {

            "test_index": index,

            "image_path": image_path,

            "true_label": true_label,

            "true_class": (
                "Seizure"
                if
                true_label == 1
                else
                "Non-seizure"
            ),

            "raw_model_output": raw_output,

            "normalized_prediction": (
                prediction
                if
                prediction is not None
                else
                "INVALID"
            ),

            "predicted_class": (
                "Seizure"
                if
                prediction == 1
                else
                "Non-seizure"
                if
                prediction == 0
                else
                "INVALID"
            )

        }

    )


    # Backup every 25 images.

    if (
        (index + 1) % 25 == 0

        or

        (index + 1)
        ==
        len(
            testing_df
        )
    ):

        pd.DataFrame(
            test_rows
        ).to_csv(

            FINAL_PREDICTIONS_CSV,

            index=False
        )


# ==================================================================================================
# 62. FINAL METRICS
# ==================================================================================================

final_metrics, metric_predictions = calculate_binary_metrics(

    test_true_labels,

    test_predictions
)


final_prediction_df = pd.DataFrame(
    test_rows
)


final_prediction_df[
    "metric_prediction"
] = metric_predictions


final_prediction_df[
    "correct"
] = (

    final_prediction_df[
        "true_label"
    ].to_numpy()

    ==

    metric_predictions
)


# Preserve test metadata too.

original_test_metadata = testing_df.drop(

    columns=[
        "_image_path",
        "_label"
    ],

    errors="ignore"

).add_prefix(
    "metadata_"
)


final_prediction_df = pd.concat(

    [

        final_prediction_df.reset_index(
            drop=True
        ),

        original_test_metadata.reset_index(
            drop=True
        )

    ],

    axis=1
)


final_prediction_df.to_csv(

    FINAL_PREDICTIONS_CSV,

    index=False
)


# ==================================================================================================
# 63. TEST TIME
# ==================================================================================================

final_test_minutes = (

    time.time()

    -

    test_start_time

) / 60.0


# ==================================================================================================
# 64. CONFUSION MATRIX CSV
# ==================================================================================================

cm_df = pd.DataFrame(

    [

        [
            final_metrics["TN"],
            final_metrics["FP"]
        ],

        [
            final_metrics["FN"],
            final_metrics["TP"]
        ]

    ],

    index=[

        "Actual_Non_Seizure",
        "Actual_Seizure"

    ],

    columns=[

        "Predicted_Non_Seizure",
        "Predicted_Seizure"

    ]
)


cm_df.to_csv(
    CONFUSION_MATRIX_CSV
)


# ==================================================================================================
# ==================================================================================================
# 65. ADD EXPERIMENT INFORMATION
# ==================================================================================================

final_metrics["Experiment"] = "R4"
final_metrics["Patient"] = 24
final_metrics["Base_Model"] = MODEL_ID
final_metrics["Representation"] = "A1-coefficient fixed-scale EEG variance topomap"
final_metrics["Method"] = "LoReFT"
final_metrics["Quantization"] = "None"
final_metrics["LoReFT_Rank"] = LOREFT_RANK
final_metrics["LoReFT_Target_Layers_Human"] = LOREFT_TARGET_LAYERS_HUMAN
final_metrics["LoReFT_Target_Layer_Python_Indices"] = LOREFT_TARGET_LAYER_INDICES
final_metrics["LoReFT_Number_Interventions"] = len(LOREFT_TARGET_LAYER_INDICES)
final_metrics["LoReFT_Prefix_Prompt_Tokens"] = LOREFT_PREFIX_TOKENS
final_metrics["LoReFT_Suffix_Prompt_Tokens"] = LOREFT_SUFFIX_TOKENS
final_metrics["LoReFT_Dropout"] = LOREFT_DROPOUT
final_metrics["Gradient_Checkpointing"] = False
final_metrics["LoReFT_Train_Dtype"] = "float32"
final_metrics["Best_Checkpoint_Directory"] = BEST_ADAPTER_DIR
final_metrics["Best_Epoch"] = best_epoch
final_metrics["Best_Validation_Balanced_Accuracy"] = best_validation_balanced_accuracy
final_metrics["Best_Validation_Loss"] = best_validation_loss
final_metrics["Trainable_Parameters"] = trainable_parameter_count
final_metrics["Total_Parameters"] = total_parameter_count
final_metrics["Trainable_Percentage"] = trainable_percentage
final_metrics["Training_Time_Minutes"] = total_training_minutes
final_metrics["Final_Test_Time_Minutes"] = final_test_minutes
final_metrics["Peak_Training_GPU_GB"] = peak_training_memory_gb

# 66. SAVE FINAL METRICS
# ==================================================================================================

with open(

    FINAL_METRICS_JSON,

    "w",

    encoding="utf-8"

) as file:

    json.dump(

        final_metrics,

        file,

        indent=4
    )


# ==================================================================================================
# 67. UPDATE CONFIG
# ==================================================================================================

experiment_config[
    "best_epoch"
] = best_epoch


experiment_config[
    "best_validation_balanced_accuracy"
] = best_validation_balanced_accuracy


experiment_config[
    "best_validation_loss"
] = best_validation_loss


experiment_config[
    "training_time_minutes"
] = total_training_minutes


experiment_config[
    "actual_epochs_run"
] = len(
    training_history
)


experiment_config[
    "stopped_early"
] = stopped_early


experiment_config[
    "early_stop_epoch"
] = early_stop_epoch


experiment_config[
    "early_stopping_patience"
] = EARLY_STOPPING_PATIENCE


experiment_config[
    "final_test_time_minutes"
] = final_test_minutes


with open(

    EXPERIMENT_CONFIG_JSON,

    "w",

    encoding="utf-8"

) as file:

    json.dump(

        experiment_config,

        file,

        indent=4
    )


# ==================================================================================================
# ==================================================================================================
# 68. FINAL REPORT
# ==================================================================================================

print("\n" + "=" * 100)
print("PATIENT 08 A1 - MEDGEMMA-4B LoReFT R4 FINAL RESULTS")
print("=" * 100)

print("\nLoReFT R4 CONFIGURATION")
print(f"Rank                 : {LOREFT_RANK}")
print(f"Language layers      : {NUM_LANGUAGE_LAYERS}")
print(f"Target human layers  : {LOREFT_TARGET_LAYERS_HUMAN}")
print(f"Target Python indices: {LOREFT_TARGET_LAYER_INDICES}")
print(f"Interventions        : {len(LOREFT_TARGET_LAYER_INDICES)}")
print(f"Prefix prompt tokens : {LOREFT_PREFIX_TOKENS}")
print(f"Suffix prompt tokens : {LOREFT_SUFFIX_TOKENS}")
print(f"Dropout              : {LOREFT_DROPOUT}")
print("Gradient checkpoint : OFF")
print("LoReFT parameter dtype: float32")
print("LoReFT compute dtype : float32")
print("Initialization        : identity/no-op")
print("Rotation weight decay : 0.0")
print(f"Source weight decay   : {WEIGHT_DECAY}")
print(f"Learning rate        : {LEARNING_RATE}")
print(f"Epochs               : {NUM_EPOCHS}")
print(f"Early stop patience  : {EARLY_STOPPING_PATIENCE}")
print(f"Actual epochs run    : {len(training_history)}")
print(f"Stopped early        : {stopped_early}")

print("\nDATA")
print(
    f"Training             : {int((actual_train_df['_label'] == 1).sum())} seizure + "
    f"{int((actual_train_df['_label'] == 0).sum())} non-seizure = {len(actual_train_df)}"
)
print(
    f"Validation           : {int((validation_df['_label'] == 1).sum())} seizure + "
    f"{int((validation_df['_label'] == 0).sum())} non-seizure = {len(validation_df)}"
)
print(
    f"Final test           : {int((testing_df['_label'] == 1).sum())} seizure + "
    f"{int((testing_df['_label'] == 0).sum())} non-seizure = {len(testing_df)}"
)

print("\nCONFUSION MATRIX")
print(f"TN = {final_metrics['TN']}")
print(f"FP = {final_metrics['FP']}")
print(f"FN = {final_metrics['FN']}")
print(f"TP = {final_metrics['TP']}")

print("\nFINAL METRICS")
print(f"Sensitivity          : {final_metrics['Sensitivity']:.4f}")
print(f"Specificity          : {final_metrics['Specificity']:.4f}")
print(f"Balanced Accuracy    : {final_metrics['Balanced_Accuracy']:.4f}")
print(f"Accuracy             : {final_metrics['Accuracy']:.4f}")
print(f"Precision            : {final_metrics['Precision']:.4f}")
print(f"F1                   : {final_metrics['F1']:.4f}")
print(f"Invalid outputs      : {final_metrics['Invalid_Predictions']}")

print("\nPARAMETER EFFICIENCY")
print(f"Trainable parameters : {trainable_parameter_count:,}")
print(f"Total parameters     : {total_parameter_count:,}")
print(f"Trainable percentage : {trainable_percentage:.6f}%")

print("\nBEST CHECKPOINT")
print(f"Best epoch           : {best_epoch}")
print(f"Best validation BA   : {best_validation_balanced_accuracy:.4f}")
print(f"Best validation loss : {best_validation_loss:.6f}")

print("\nTIME")
print(f"Training time        : {total_training_minutes:.2f} minutes")
print(f"Final test time      : {final_test_minutes:.2f} minutes")

print("\n" + "=" * 100)
print("R4 FILES SAVED")
print("=" * 100)
print("\nBest LoReFT checkpoint:")
print(BEST_ADAPTER_DIR)
print("\nTraining split:")
print(TRAIN_SPLIT_CSV)
print("\nValidation split:")
print(VALIDATION_SPLIT_CSV)
print("\nTraining history:")
print(TRAINING_HISTORY_CSV)
print("\nFinal test predictions:")
print(FINAL_PREDICTIONS_CSV)
print("\nFinal metrics:")
print(FINAL_METRICS_JSON)
print("\nConfusion matrix:")
print(CONFUSION_MATRIX_CSV)
print("\nExperiment configuration:")
print(EXPERIMENT_CONFIG_JSON)
print("\nLoReFT target information:")
print(TARGET_MODULES_TXT)
print("\n" + "=" * 100)
print("PATIENT 08 A1 MEDGEMMA-4B LoReFT R4 COMPLETE")
print("=" * 100)
